# 📊 Phase 01B — GPR Index · Historical News Collection · Sentiment · Regime Events  [v7]
### Research Project: Regime Detection — NIFTY50 · BANKNIFTY · NIFTYNEXT50
**Author:** Swagato M25AI2107 | **Drive Folder:** `MyDrive/Regime_Detection_Swagato_M25AI2107/Data/`

---

## 🗺️ What This Notebook Does

This notebook is the **data-acquisition and feature-engineering backbone** of the regime-detection research pipeline.  
It runs end-to-end to produce a single master feature table (`gpr_combined_monthly.csv`) that Phase 02 uses to train regime classifiers for Indian equity indices.

| # | Module | What it collects | Output file |
|---|--------|-----------------|-------------|
| 1 | **[DRIVE]** | Mount Google Drive | — |
| 2 | **[LIBS]** | Install & import all packages | — |
| 3 | **[CFG]** | API keys, paths, constants, keyword lists | — |
| 4 | **[GPR-INDEX]** | Caldara & Iacoviello Geopolitical Risk Index | `raw/gpr/gpr_monthly.csv` |
| 5 | **[CPU]** | Climate-Policy Uncertainty Index (§4.2.6) | `raw/climate_policy/cpu_monthly.csv` |
| 6 | **[NEWS-RSS]** | RSS feeds from 27 financial news sources | `raw/rss/news_raw_YYYYMM.csv` |
| 7 | **[YFINANCE]** | Yahoo Finance ticker news | `raw/yfinance/yf_news_YYYYMM.csv` |
| 8 | **[GDELT-2.0]** | GDELT 2.0 Doc API (Feb 2015 → 2026) | `raw/gdelt/gdelt2_YYYY-MM.csv` |
| 9 | **[GDELT-1.0]** | GDELT 1.0 raw CSV events (2000 → Feb 2015) | `raw/gdelt/gdelt1_monthly_tone.csv` |
| 10 | **[ALPHAVANTAGE]** | Pre-scored news sentiment (~2 years rolling) | `raw/alphavantage/av_monthly_sentiment.csv` |
| 11 | **[WAYBACK]** | Wayback Machine CDX archived titles (2008–2015) | `raw/wayback/wayback_titles.csv` |
| 12 | **[MERGE-RAW]** | Unify all sources into one corpus | `cleaned/news_corpus_all.csv` |
| 13 | **[FINBERT]** | FinBERT sentiment on full corpus | `raw/sentiment/finbert_checkpoint.csv` |
| 14 | **[RISK-CATEGORY]** | Keyword-based article classification | `raw/sentiment/news_sentiment_monthly.csv` |
| 15 | **[REGIME-EVENTS]** | Live-fetched regime-change events | `raw/regime_events/regime_events.csv` |
| 16 | **[FRED]** | 23 India macroeconomic series from FRED | `raw/fred/fred_india_macro.csv` |
| 16B | **[MARKET-DATA]** | India VIX · USD/INR · Gold historical prices | `raw/other_market_data/other_market_data_combined.csv` |
| 17 | **[VERIFY]** | Sanity-check all output files | — |
| 18 | **[COMBINE]** | Merge everything into master feature table | `cleaned/gpr_combined_monthly.csv` |

---

## 🧱 Design Principles

- **Open-source only** — sentiment uses FinBERT (Apache 2.0), no paid LLM API calls  
- **Incremental / crash-safe** — every expensive loop writes per-month CSVs so a Colab restart resumes from where it left off  
- **Politically neutral** — regime events record only *event types* (election, budget, crisis), never party names or leaders  
- **Coverage strategy** — GDELT is the backbone (2015–2026); Wayback extends back to 2008; GDELT 1.0 fills 2000–2015 via tone proxy
- **Market signals** — India VIX (fear gauge), USD/INR (currency stress), and Gold (safe-haven demand) added as complementary regime indicators

---

## 📂 Output Folder Layout

```
Data/
├── raw/
│   ├── gpr/                    ← Geopolitical Risk Index CSVs
│   ├── rss/                    ← RSS-scraped articles (monthly slices + master)
│   ├── yfinance/               ← yfinance ticker news (monthly + master)
│   ├── gdelt/                  ← GDELT 1.0 tone + GDELT 2.0 article files
│   ├── alphavantage/           ← Alpha Vantage per-topic + monthly sentiment
│   ├── wayback/                ← Wayback CDX archived titles
│   ├── fred/                   ← FRED macro data + annual slices
│   ├── regime_events/          ← Regime events CSVs
│   ├── climate_policy/         ← CPU index + engineered features
│   ├── sentiment/              ← FinBERT checkpoint + monthly tables
│   └── other_market_data/      ← India VIX, USD/INR, Gold monthly CSVs
└── cleaned/
    ├── news_corpus_all.csv            ← Unified deduplicated news corpus from all 5 sources — 2000–2026
    └── news_corpus_all_sentiment.csv  ← Same corpus with FinBERT sentiment scores added
    └── gpr_combined_monthly.csv       ← Master feature table — one row per month, all features
```


---
## Section 1 · [DRIVE] Mount Google Drive

**Why?**  
All raw data, checkpoints, and outputs are stored on Google Drive so they persist across Colab sessions.  
Without mounting the drive, every kernel restart would wipe the data.


In [1]:
from google.colab import drive
drive.mount('/content/drive')
print('✅ Google Drive mounted successfully.')

Mounted at /content/drive
✅ Google Drive mounted successfully.


---
## Section 2 · [LIBS] Install & Import Libraries

**Why?**  
Colab environments are ephemeral — packages not pre-installed on the VM must be installed at runtime.

| Package group | Purpose |
|---------------|---------|
| `gdeltdoc` | Python wrapper for GDELT 2.0 Doc API |
| `transformers`, `torch`, `sentencepiece` | FinBERT NLP sentiment model |
| `feedparser`, `beautifulsoup4`, `lxml` | RSS feed parsing and HTML scraping |
| `fredapi` | Fetch macroeconomic series from St. Louis FRED |
| `yfinance` | Yahoo Finance news and market data |
| `tqdm` | Progress bars for long-running loops |
| `openpyxl`, `xlrd` | Read Excel files (GPR index downloads as `.xls`) |
| `wikipedia-api` | Fetch structured data from Wikipedia (regime events) |


In [2]:
# ── Install all required packages ─────────────────────────────────────────────
# -q suppresses verbose output; packages only need to install once per session.
!pip install -q gdeltdoc transformers torch sentencepiece
!pip install -q feedparser requests beautifulsoup4 lxml tqdm
!pip install -q pandas openpyxl xlrd wikipedia-api python-dateutil
!pip install -q yfinance fredapi
print('✅ All packages installed.')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 64.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.3 which is incompatible.
gradio 5.50.0 requires pandas<3.0,>=1.0, but you have pandas 3.0.3 which is incompatible.
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.5/81.5 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.2/129.2 kB 6.2 MB/s eta 0:00:00
✅ All packages installed.


In [3]:
# ── Standard library imports ───────────────────────────────────────────────────
import os, re, time, json, logging, warnings
import io, zipfile, calendar
from datetime import datetime, date, timedelta
from dateutil.relativedelta import relativedelta
from pathlib import Path
from io import BytesIO
from concurrent.futures import ThreadPoolExecutor, as_completed

# ── Third-party imports ────────────────────────────────────────────────────────
import requests
import feedparser
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
from tqdm.auto import tqdm

# ── NLP / ML ──────────────────────────────────────────────────────────────────
import torch
from transformers import pipeline

# ── GDELT ─────────────────────────────────────────────────────────────────────
from gdeltdoc import GdeltDoc, Filters

# ── Market data ───────────────────────────────────────────────────────────────
import yfinance as yf

# ── Suppress non-critical warnings to keep output readable ────────────────────
warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.WARNING)

print('✅ All libraries imported.')
print(f'   PyTorch version : {torch.__version__}')
print(f'   CUDA available  : {torch.cuda.is_available()}')

✅ All libraries imported.
   PyTorch version : 2.11.0+cpu
   CUDA available  : False


---
## Section 3 · [CFG] Global Configuration

**Why?**  
Centralising all API keys, file paths, date ranges, keyword lists, and hyperparameters here means:
1. You only ever edit **one cell** to customise the pipeline.
2. Downstream cells reference constants — no magic strings scattered around.
3. The config block doubles as documentation of every tunable parameter.

> - Alpha Vantage: https://www.alphavantage.co/support/#api-key  
> - FRED: https://fred.stlouisfed.org/docs/api/api_key.html


In [4]:
# ══════════════════════════════════════════════════════════════════════════════
# 3A. SOURCE URLS
# ══════════════════════════════════════════════════════════════════════════════
# The GPR global index is published as an Excel file by Matteo Iacoviello. We download it fresh each run to pick up the latest monthly observations.
GPR_GLOBAL_URL = 'https://www.matteoiacoviello.com/gpr_files/data_gpr_export.xls'
# GPR_COUNTRY_URL = 'https://www.matteoiacoviello.com/gpr_files/data_gpr_country_export.xls'

# ══════════════════════════════════════════════════════════════════════════════
# 3B. GOOGLE DRIVE PATH TREE
# ══════════════════════════════════════════════════════════════════════════════
# All outputs are written under DATA_ROOT so the full dataset persists across Colab sessions.
GDRIVE_BASE   = '/content/drive/MyDrive/Regime_Detection_Swagato_M25AI2107'
DATA_ROOT     = os.path.join(GDRIVE_BASE, 'Data')

RAW           = os.path.join(DATA_ROOT, 'raw')
GPR_DIR       = os.path.join(RAW, 'gpr')
RSS_DIR       = os.path.join(RAW, 'rss')
YF_DIR        = os.path.join(RAW, 'yfinance')
GDELT_DIR     = os.path.join(RAW, 'gdelt')
AV_DIR        = os.path.join(RAW, 'alphavantage')
WAYBACK_DIR   = os.path.join(RAW, 'wayback')
FRED_DIR      = os.path.join(RAW, 'fred')
EVENTS_DIR    = os.path.join(RAW, 'regime_events')
SENTIMENT_DIR = os.path.join(RAW, 'sentiment')
CPU_DIR       = os.path.join(RAW, 'climate_policy')
MARKET_DIR    = os.path.join(RAW, 'other_market_data')
CLEANED_DIR   = os.path.join(DATA_ROOT, 'cleaned')

# Create all folders (no-op if they already exist)
for d in [GPR_DIR, RSS_DIR, YF_DIR, GDELT_DIR, AV_DIR, WAYBACK_DIR, FRED_DIR, EVENTS_DIR, SENTIMENT_DIR, CPU_DIR, MARKET_DIR, CLEANED_DIR]:
    os.makedirs(d, exist_ok=True)
print('✅ Folder tree verified / created.')

# ══════════════════════════════════════════════════════════════════════════════
# 3C. API KEYS
# ══════════════════════════════════════════════════════════════════════════════
AV_API_KEY   = '28MM0KMC8DHUP5SQ'                    # Free at alphavantage.co
FRED_API_KEY = 'a0591c3b8838bac1ddcf95d624eccc31'    # Free at fred.stlouisfed.org

# ══════════════════════════════════════════════════════════════════════════════
# 3D. DATE RANGES
# ══════════════════════════════════════════════════════════════════════════════
# The pipeline covers 2000-01 → 2026-03 (the full NIFTY 50 listed history).
# GDELT 2.0 only goes back to Feb 2015; GDELT 1.0 is saved separately as a monthly tone source.
GPR_START      = '2000-01'
GPR_END        = '2026-03'
GDELT1_START   = date(2000, 1, 1)
GDELT1_END     = date(2015, 1, 31)
GDELT2_START   = date(2015, 2, 1)
GDELT2_END     = date(2026, 3, 31)
YF_SCRAPE_FROM = '2020-01'     # yfinance only reliably covers ~5 years rolling
MKT_START      = GPR_START + '-01'  # Market data fetch start (e.g. '2000-01-01')
MKT_END        = GPR_END   + '-28'  # Market data fetch end   (e.g. '2026-03-28')

# ══════════════════════════════════════════════════════════════════════════════
# 3E. FINBERT HYPERPARAMETERS
# ══════════════════════════════════════════════════════════════════════════════
# ProsusAI/finbert is a BERT model fine-tuned on financial text. BATCH_SIZE=32 fits Colab T4; reduce to 8 if you get OOM errors on the free tier.
FINBERT_MODEL = 'ProsusAI/finbert'
BATCH_SIZE    = 32
MAX_TOKENS    = 512

# ══════════════════════════════════════════════════════════════════════════════
# 3F. HTTP HEADERS
# ══════════════════════════════════════════════════════════════════════════════
# A descriptive User-Agent makes our scraper identifiable and avoids bot blocks.
HEADERS = {'User-Agent': 'Mozilla/5.0 (compatible; RegimeResearch/2.0)'}

# ══════════════════════════════════════════════════════════════════════════════
# 3G. RELEVANCE FILTER — shared across all scrapers
# ══════════════════════════════════════════════════════════════════════════════
# Articles that do NOT contain any of these terms are discarded before storage. This keeps the corpus focused on Indian equity markets and macro events.
INDEX_TERMS = [
    # Core indices
    'nifty', 'sensex', 'banknifty', 'bank nifty', 'nifty 50', 'nifty next 50',
    # Market language
    'stock market', 'equity market', 'markets', 'benchmark index',
    'stocks', 'shares', 'indices', 'market rally', 'market crash',
    'market close', 'market open',
    # India-specific
    'nse', 'bse', 'dalal street', 'indian market',
    # Capital flows
    'fii', 'dii', 'foreign investors', 'capital inflow', 'outflow',
    # Monetary policy
    'rbi', 'repo rate', 'inflation', 'cpi', 'monetary policy',
    # FX / commodities
    'rupee', 'usd inr', 'crude oil', 'brent oil',
    # Global macro
    'fed', 'federal reserve', 'china', 'us economy', 'recession',
    # Geopolitics
    'war', 'conflict', 'geopolitical', 'tensions', 'sanctions',
    'military', 'attack', 'crisis',
]

def _is_relevant(text: str) -> bool:
    """Return True if text contains at least one relevance keyword."""
    t = text.lower()
    return any(term in t for term in INDEX_TERMS)

# ══════════════════════════════════════════════════════════════════════════════
# 3H. GDELT KEYWORD QUERIES
# ══════════════════════════════════════════════════════════════════════════════
# Each string becomes a separate GDELT 2.0 full-text query.
GDELT_QUERIES = [
    'Nifty 50', 'Bank Nifty', 'Nifty Next 50', 'India stock market',
    'India banking sector', 'India financial markets', 'NSE India',
    'Bombay Stock Exchange', 'Indian economy', 'Reserve Bank of India',
    'RBI policy', 'SEBI regulation', 'India GDP growth', 'India inflation',
    'Indian rupee', 'foreign investors India', 'India election',
    'India budget', 'India monsoon', 'India flood drought', 'India geopolitical',
    # Climate-Risk News Share
    'India climate change', 'India carbon emission', 'India extreme weather',
    'India climate risk', 'India transition risk', 'India net zero',
    'India Paris Agreement', 'India COP climate',
    # Green-Transition Event Density
    'India renewable energy', 'India solar power', 'India wind energy',
    'India electric vehicle', 'India green hydrogen', 'India energy transition',
    'India decarbonization', 'India clean energy policy', 'India green bond',
    # ESG-Linked Sentiment
    'India ESG', 'India ESG investing', 'India BRSR', 'India sustainable finance',
    'India green finance', 'India climate finance',
]

# ══════════════════════════════════════════════════════════════════════════════
# 3I. ALPHA VANTAGE TOPIC BUCKETS
# ══════════════════════════════════════════════════════════════════════════════
# AV's NEWS_SENTIMENT endpoint filters by predefined topic categories.
AV_TOPICS = [
    'economy_macro', 'financial_markets', 'ipo',
    'mergers_and_acquisitions', 'monetary_policy', 'geopolitical',
    'energy_transportation',   # Covers EV / oil / renewable energy news
    'sustainability',          # ESG / green-transition / climate-policy news
]

# ══════════════════════════════════════════════════════════════════════════════
# 3J. YFINANCE TICKERS
# ══════════════════════════════════════════════════════════════════════════════
# These tickers map to the three indices studied in the research.
YF_TICKERS = [
    '^NSEI',        # NIFTY 50
    '^NSEBANK',     # BANK NIFTY
    '^NSMIDCP',     # NIFTY NEXT 50 proxy
    'NIFTYBEES.NS', # ETF tracking NIFTY 50
    'BANKBEES.NS',  # ETF tracking BANK NIFTY
    'JUNIORBEES.NS',# ETF tracking NIFTY NEXT 50
]

# ══════════════════════════════════════════════════════════════════════════════
# 3K. RISK CATEGORY KEYWORD LISTS
# ══════════════════════════════════════════════════════════════════════════════
# 'financial' and 'political' are the primary regime-driver categories. 'climate', 'green_transition', 'esg' are the sustainability block.
RISK_KEYWORDS = {
    'financial': [
        'nifty', 'sensex', 'banknifty', 'nse', 'bse', 'rbi', 'sebi', 'repo rate',
        'inflation', 'gdp', 'fiscal deficit', 'current account', 'rupee', 'fii',
        'dii', 'ipo', 'credit', 'liquidity', 'bond yield', 'fed rate', 'interest rate',
        'earnings', 'quarterly result', 'profit', 'revenue', 'stock market',
        'equity', 'derivative', 'futures', 'options', 'rally', 'correction',
    ],
    'political': [
        'election', 'lok sabha', 'rajya sabha', 'parliament', 'cabinet', 'minister',
        'policy reform', 'budget', 'union budget', 'tax reform', 'gst',
        'geopolitical', 'border', 'sanction', 'trade war', 'tariff', 'bilateral',
        'imf', 'world bank', 'g20', 'un', 'war', 'conflict', 'coup',
        'regulatory', 'disinvestment', 'privatisation',
    ],
    # Physical climate risk → Climate-Risk News Share
    'climate': [
        'monsoon', 'flood', 'drought', 'cyclone', 'earthquake', 'disaster',
        'climate change', 'global warming', 'carbon', 'emission', 'carbon tax',
        'carbon credit', 'carbon market', 'cap and trade', 'paris agreement',
        'cop26', 'cop27', 'cop28', 'cop29', 'ipcc', 'net zero', 'net-zero',
        'heatwave', 'extreme weather', 'temperature rise', 'sea level',
        'agriculture', 'kharif', 'rabi', 'crop failure', 'el nino', 'la nina',
        'imd', 'rainfall', 'water scarcity', 'deforestation', 'biodiversity',
        'climate risk', 'transition risk', 'stranded asset',
    ],
    # Green-Transition Event Density
    'green_transition': [
        'renewable energy', 'solar', 'solar power', 'solar panel', 'wind energy',
        'wind farm', 'offshore wind', 'electric vehicle', 'ev', 'ev adoption',
        'battery storage', 'lithium', 'rare earth', 'green hydrogen',
        'decarbonization', 'decarbonisation', 'net-zero pledge', 'carbon neutral',
        'energy transition', 'clean energy', 'green energy', 'clean tech',
        'national solar mission', 'pm surya ghar', 'pm kusum', 'green taxonomy',
        'sustainable finance', 'green bond', 'sustainability bond', 'esg bond',
        'iea', 'irena', 'renewable capacity', 'mw solar', 'gw renewable',
        'rooftop solar', 'smart grid', 'energy storage',
    ],
    # ESG-Linked Sentiment — polarity from ESG/sustainability news
    'esg': [
        'esg', 'environmental social governance', 'esg rating', 'esg score',
        'esg fund', 'esg investing', 'sustainable investing', 'sri', 'impact investing',
        'green finance', 'responsible investment', 'climate finance',
        'carbon footprint', 'scope 1', 'scope 2', 'scope 3', 'ghg disclosure',
        'sebi esg', 'brsr', 'tcfd', 'csrd', 'eu taxonomy', 'greenwashing',
        'esg compliance', 'esg risk', 'esg report', 'sustainability report',
        'esg benchmark', 'msci esg', 'ftse4good', 'dow jones sustainability',
    ],
}

# Flat ESG keyword set — used in the ESG Sentiment Score formula:
ESG_KEYWORDS = set(RISK_KEYWORDS['esg'] + RISK_KEYWORDS['green_transition'])

# ══════════════════════════════════════════════════════════════════════════════
# 3L. RSS FEED REGISTRY
# ══════════════════════════════════════════════════════════════════════════════
# 27 feeds covering: Indian financial press, RBI/SEBI official releases, climate/ESG publishers, global macro, central banks, commodities, geopolitics.
NEWS_FEEDS = {
    # ── India — Financial / Markets ──
    'ET_Markets'       : 'https://economictimes.indiatimes.com/markets/rssfeeds/1977021501.cms',
    'MoneyControl_Mkt' : 'https://www.moneycontrol.com/rss/MCtopnews.xml',
    'LiveMint_Mkt'     : 'https://www.livemint.com/rss/markets',
    'Business_Standard': 'https://www.business-standard.com/rss/markets-106.rss',
    'CNBC_India'       : 'https://www.cnbctv18.com/commonfeeds/v1/eng/rss/market.xml',
    'FinancialExpress' : 'https://www.financialexpress.com/market/feed/',
    'EconomicTimes'    : 'https://economictimes.indiatimes.com/rssfeedsdefault.cms',
    # ── India — Macro / Policy ──
    'ET_Economy'       : 'https://economictimes.indiatimes.com/news/economy/rssfeeds/1373380680.cms',
    'RBI_PressReleases': 'https://www.rbi.org.in/scripts/rss.aspx',
    'SEBI_Circulars'   : 'https://www.sebi.gov.in/sebi_data/rss/sebilatest.xml',
    'PIB_Economy'      : 'https://pib.gov.in/rss.aspx?mincode=15',
    # ── Climate / ESG ──
    'DownToEarth'      : 'https://www.downtoearth.org/rss',
    'ET_Climate'       : 'https://economictimes.indiatimes.com/environment/rssfeeds/22419226.cms',
    # ── Global — Markets / Macro ──
    'Reuters_Business' : 'https://feeds.reuters.com/reuters/businessNews',
    'Reuters_Markets'  : 'https://feeds.reuters.com/reuters/marketsNews',
    'Bloomberg_Asia'   : 'https://feeds.bloomberg.com/markets/news.rss',
    'Bloomberg_Economy': 'https://feeds.bloomberg.com/economics/news.rss',
    'CNBC_World'       : 'https://www.cnbc.com/id/100003114/device/rss/rss.html',
    'WSJ_World'        : 'https://feeds.a.dj.com/rss/RSSWorldNews.xml',
    # ── Central Banks ──
    'FederalReserve'   : 'https://www.federalreserve.gov/feeds/press_all.xml',
    'ECB'              : 'https://www.ecb.europa.eu/rss/press.html',
    'IMF_News'         : 'https://www.imf.org/en/News/rss',
    # ── Commodities / Energy ──
    'OilPrice'         : 'https://oilprice.com/rss/main',
    'Reuters_Energy'   : 'https://feeds.reuters.com/reuters/energyNews',
    # ── Geopolitics ──
    'Reuters_World'    : 'https://feeds.reuters.com/Reuters/worldNews',
    'BBC_World'        : 'http://feeds.bbci.co.uk/news/world/rss.xml',
    'AlJazeera'        : 'https://www.aljazeera.com/xml/rss/all.xml',
}

# ══════════════════════════════════════════════════════════════════════════════
# 3M. WAYBACK MACHINE SETTINGS
# ══════════════════════════════════════════════════════════════════════════════
# These domains are crawled via the CDX API to backfill Indian news 2006–2015.
WAYBACK_DOMAINS = {
    'economictimes.indiatimes.com': 2008,
    'livemint.com'                : 2009,
    'business-standard.com'       : 2007,
    'moneycontrol.com'            : 2008,
    'thehindubusinessline.com'    : 2006,
}
WAYBACK_SAMPLE_PER_MONTH = 30   # Max articles to fetch per (domain, month)
MAX_WORKERS               = 8   # Threads for parallel HTTP requests

# ══════════════════════════════════════════════════════════════════════════════
# 3N. FRED SERIES REGISTRY (23 India macroeconomic series)
# ══════════════════════════════════════════════════════════════════════════════
# Monthly series are pulled at native frequency.
# Quarterly series are forward-filled to month-end for alignment with news data.
FRED_SERIES_MONTHLY = {
    'INDIRLTLT01STM'  : 'rate_10y_bond_yield',
    'INDIR3TIB01STM'  : 'rate_3m_interbank',
    'RBINBIS'         : 'fx_reer',
    'CCUSMA02INM618N' : 'fx_usdinr_avg',
    'CPALTT01INM659N' : 'inf_cpi_yoy',
    'WPIATT01INM661N' : 'inf_wpi_index',
    'MABMM301INM189N' : 'mon_m3_supply',
    'MANMM101INM657S' : 'mon_m1_growth_mom',
    'INDPROMANMISMEI' : 'act_mfg_production_sa',
    'INDPRINTO01GYSAM': 'act_iip_growth_yoy',
    'INDLOLITOAASTSAM': 'lead_oecd_cli_amp_adj',
    'XTEXVA01INM667S' : 'trade_exports_usd_sa',
    'XTIMVA01INM667S' : 'trade_imports_usd_sa',
    'SPASTT01INM657N' : 'mkt_share_price_growth_mom',
}
FRED_SERIES_QUARTERLY = {
    'INDGDPRQPSMEI'   : 'gdp_real_growth_yoy',
    'CRDQINAPABIS'    : 'credit_private_nonfin_inr',
    'QINCAM770A'      : 'credit_total_nonfin_pct_gdp',
    'INDB6FATT01CXCUQ': 'bop_financial_acct_net_usd',
    'BSCURT02INQ160S' : 'biz_capacity_util_mfg',
    'BSCICP02INQ460S' : 'biz_confidence_mfg',
    'NAEXKP02INQ659S' : 'gdp_private_consumption_yoy',
    'NAEXKP04INQ659S' : 'gdp_gfcf_yoy',
    'QINR628BIS'      : 'asset_residential_property_price',
}

# ── Configuration Summary ──────────────────────────────────────────────────────
print('✅ [CFG] Configuration loaded.')
print(f'   Project date range : {GPR_START} → {GPR_END}')
print(f'   GDELT 2.0 window   : {GDELT2_START} → {GDELT2_END}')
print(f'   GDELT 1.0 window   : {GDELT1_START} → {GDELT1_END}')
print(f'   FinBERT model      : {FINBERT_MODEL}')
print(f'   RSS feeds          : {len(NEWS_FEEDS)}')
print(f'   yfinance tickers   : {len(YF_TICKERS)} | from {YF_SCRAPE_FROM}')
print(f'   AV API key         : {"SET" if bool(AV_API_KEY) else "NOT SET ⚠️"}')
print(f'   FRED API key       : {"SET" if bool(FRED_API_KEY)     else "NOT SET ⚠️"}')
print(f'   FRED monthly series: {len(FRED_SERIES_MONTHLY)}')
print(f'   FRED quarterly     : {len(FRED_SERIES_QUARTERLY)}')
print(f'   Market data window : {MKT_START} → {MKT_END}')

✅ Folder tree verified / created.
✅ [CFG] Configuration loaded.
   Project date range : 2000-01 → 2026-03
   GDELT 2.0 window   : 2015-02-01 → 2026-03-31
   GDELT 1.0 window   : 2000-01-01 → 2015-01-31
   FinBERT model      : ProsusAI/finbert
   RSS feeds          : 27
   yfinance tickers   : 6 | from 2020-01
   AV API key         : SET
   FRED API key       : SET
   FRED monthly series: 14
   FRED quarterly     : 9
   Market data window : 2000-01-01 → 2026-03-28


---
## Section 4 · [GPR-INDEX] Download Geopolitical Risk Index

**Why?**  
The **Caldara & Iacoviello (AER 2022) GPR Index** is the primary exogenous geopolitical risk signal in the model.  
It is constructed from automated text searches of 11 major newspapers and is available monthly back to 1985.  
We use it as both a feature and a regime-labelling anchor.

| Column | Description |
|--------|-------------|
| `gpr_global` | Headline GPR index (threats + acts) |
| `gpr_global_threats` | GPR sub-index: geopolitical threats only |
| `gpr_global_acts` | GPR sub-index: geopolitical acts only |
| `gpr_global_hist` | Historical GPR (1985–present) |
| `gpr_india` | Country-level India GPR |
| `gpr_india_threats` / `gpr_india_acts` | India sub-indices |

**Source:** https://www.matteoiacoviello.com/gpr.htm  
**Saves:** `raw/gpr/gpr_global_raw.xls` (raw download) · `raw/gpr/gpr_monthly.csv` (parsed)


In [5]:
# ─────────────────────────────────────────────────────────────────────────────
# GPR HELPER FUNCTIONS
# ─────────────────────────────────────────────────────────────────────────────

def download_gpr(url: str, label: str, save_path: str):
    """Download the GPR Excel file and return as a DataFrame.

    We save the raw .xls bytes to disk for traceability so the exact
    downloaded file is always available for audit.
    """
    try:
        print(f'⬇️  Downloading {label}...')
        resp = requests.get(url, headers=HEADERS, timeout=60)
        resp.raise_for_status()
        with open(save_path, 'wb') as f:
            f.write(resp.content)
        df = pd.read_excel(BytesIO(resp.content), engine='xlrd')
        print(f'   ✅ {label}: shape={df.shape} | saved → {save_path}')
        return df
    except Exception as e:
        print(f'❌ Failed to download {label}: {e}')
        return None


def parse_global_gpr(df):
    """Extract and rename global GPR columns from the raw Excel.

    The raw file uses short uppercase codes (GPR, GPRT, GPRA, …).
    We rename them to descriptive snake_case names for downstream clarity.
    """
    if df is None:
        return None
    df = df.copy()
    df['period'] = pd.to_datetime(df['month']).dt.to_period('M')
    df = df.set_index('period').sort_index()

    rename_map = {}
    for col in df.columns:
        c = col.upper().strip()
        if   c == 'GPR'       : rename_map[col] = 'gpr_global'
        elif c == 'GPRT'      : rename_map[col] = 'gpr_global_threats'
        elif c == 'GPRA'      : rename_map[col] = 'gpr_global_acts'
        elif c == 'GPRH'      : rename_map[col] = 'gpr_global_hist'
        elif c == 'GPRHT'     : rename_map[col] = 'gpr_global_hist_threats'
        elif c == 'GPRHA'     : rename_map[col] = 'gpr_global_hist_acts'
        elif c == 'GPR_NOEW'  : rename_map[col] = 'gpr_global_noew'
        elif c == 'GPR_AND'   : rename_map[col] = 'gpr_global_and'
        elif c == 'GPR_BASIC' : rename_map[col] = 'gpr_global_basic'
        elif c == 'SHARE_GPR' : rename_map[col] = 'gpr_share'
        elif c == 'N10'       : rename_map[col] = 'gpr_news_count'

    df = df[list(rename_map.keys())].rename(columns=rename_map)
    print(f'   🌍 Global GPR columns: {list(df.columns)}')
    return df


def parse_india_gpr(df):
    """Extract India-specific GPR columns from the global Excel file.

    The global file also contains country-level series (GPRC_IND, GPRCT_IND, …).
    If not present (older file versions), returns None gracefully.
    """
    if df is None:
        return None
    df = df.copy()
    df['period'] = pd.to_datetime(df['month']).dt.to_period('M')
    df = df.set_index('period').sort_index()

    rename_map = {}
    for col in df.columns:
        c = col.upper().strip()
        if   c == 'GPRC_IND'  : rename_map[col] = 'gpr_india'
        elif c == 'GPRCT_IND' : rename_map[col] = 'gpr_india_threats'
        elif c == 'GPRCA_IND' : rename_map[col] = 'gpr_india_acts'
        elif c == 'GPRHC_IND' : rename_map[col] = 'gpr_india_hist'

    if not rename_map:
        print('⚠️  India GPR columns not found in the downloaded file.')
        return None

    df = df[list(rename_map.keys())].rename(columns=rename_map)
    print(f'   🇮🇳 India GPR columns: {list(df.columns)}')
    return df


In [6]:
# ─────────────────────────────────────────────────────────────────────────────
# RUN: Download, parse, merge, and save the GPR index
# ─────────────────────────────────────────────────────────────────────────────

# Download the global GPR file (also contains India sub-series)
gpr_global_raw = download_gpr(
    GPR_GLOBAL_URL, 'GPR Global',
    os.path.join(GPR_DIR, 'gpr_global_raw.xls')
)

# Parse global and India sub-series separately
gpr_global = parse_global_gpr(gpr_global_raw)
gpr_india  = parse_india_gpr(gpr_global_raw)

# Outer-join so we keep all months even if one series is shorter
if gpr_global is not None and gpr_india is not None:
    gpr_index = gpr_global.join(gpr_india, how='outer')
elif gpr_global is not None:
    gpr_index = gpr_global
else:
    raise ValueError('❌ GPR download failed — no data available.')

gpr_index = gpr_index.sort_index()

# Save to CSV for use in [COMBINE]
gpr_path = os.path.join(GPR_DIR, 'gpr_monthly.csv')
gpr_index.to_csv(gpr_path)

print(f'\n✅ GPR dataset ready: {gpr_index.shape}')
print(f'   Columns : {list(gpr_index.columns)}')
print(f'💾 Saved  : {gpr_path}')
display(gpr_index.tail(5))


⬇️  Downloading GPR Global...
   ✅ GPR Global: shape=(1517, 115) | saved → /content/drive/MyDrive/Regime_Detection_Swagato_M25AI2107/Data/raw/gpr/gpr_global_raw.xls
   🌍 Global GPR columns: ['gpr_global', 'gpr_global_threats', 'gpr_global_acts', 'gpr_global_hist', 'gpr_global_hist_threats', 'gpr_global_hist_acts', 'gpr_share', 'gpr_news_count', 'gpr_global_noew', 'gpr_global_and', 'gpr_global_basic']
   🇮🇳 India GPR columns: ['gpr_india', 'gpr_india_hist']

✅ GPR dataset ready: (1517, 13)
   Columns : ['gpr_global', 'gpr_global_threats', 'gpr_global_acts', 'gpr_global_hist', 'gpr_global_hist_threats', 'gpr_global_hist_acts', 'gpr_share', 'gpr_news_count', 'gpr_global_noew', 'gpr_global_and', 'gpr_global_basic', 'gpr_india', 'gpr_india_hist']
💾 Saved  : /content/drive/MyDrive/Regime_Detection_Swagato_M25AI2107/Data/raw/gpr/gpr_monthly.csv


,gpr_global,gpr_global_threats,gpr_global_acts,gpr_global_hist,gpr_global_hist_threats,gpr_global_hist_acts,gpr_share,gpr_news_count,gpr_global_noew,gpr_global_and,gpr_global_basic,gpr_india,gpr_india_hist
period,,,,,,,,,,,,,
2026-01,168.629868,220.171417,100.222099,128.070435,198.120316,70.965256,5.057858,16074.0,128.757904,122.399300,182.655304,0.329725,0.340823
2026-02,120.745155,157.467438,81.679771,108.260468,159.077957,74.479759,3.621611,15159.0,115.885071,103.005249,136.046768,0.435385,0.350877
2026-03,329.653564,318.030731,448.654785,252.136993,281.973450,314.048523,9.887576,17345.0,264.547119,175.395142,348.992462,1.003171,0.874331
2026-04,250.807281,268.353180,307.247925,211.172653,261.421631,241.835785,7.522674,16098.0,203.100952,149.203995,272.836090,0.583923,0.631466
2026-05,184.177048,214.394455,198.083832,161.460876,214.092087,170.862305,5.524177,14518.0,141.601242,112.011208,189.131775,0.495936,0.668395


---
## Section 5 · [CPU] Climate-Policy Uncertainty Index

**Why?**  
The **Baker-Bloom-Davis Climate Policy Uncertainty (CPU) Index** measures uncertainty  
surrounding government climate-related regulations and policy decisions.  
It is constructed from newspaper text searches — methodologically consistent with the GPR index.

**Source:** Baker, Bloom & Davis — https://www.policyuncertainty.com/climate_uncertainty.html  
**Saves:** `raw/climate_policy/cpu_monthly.csv`


In [7]:
# ─────────────────────────────────────────────────────────────────────────────
# CLIMATE POLICY UNCERTAINTY (CPU) INDEX DOWNLOAD
# ─────────────────────────────────────────────────────────────────────────────
CPU_URL  = 'https://www.policyuncertainty.com/media/cpu_pu.xlsx'
CPU_PATH = os.path.join(CPU_DIR, 'cpu_monthly.csv')
CPU_HEADERS = {'User-Agent': 'Mozilla/5.0'}  # local alias — avoids clobbering global HEADERS

# ─────────────────────────────────────────────────────────────────────────────
# DOWNLOAD CPU INDEX
# ─────────────────────────────────────────────────────────────────────────────

def download_cpu_index(url: str, save_path: str) -> pd.DataFrame:
    """Download and save the raw monthly CPU index. Derived features are created in EDA."""
    if os.path.exists(save_path):
        print(f'✅ CPU raw cache found: {save_path}')
        return pd.read_csv(save_path)
    try:
        print('⬇️ Downloading CPU index...')
        resp = requests.get(url, headers=CPU_HEADERS, timeout=60)
        resp.raise_for_status()
        df = pd.read_excel(BytesIO(resp.content),sheet_name='data')

        df.columns = [str(c).strip().lower() for c in df.columns]

        print('\n📌 Columns:')
        print(df.columns.tolist())

        df = df.dropna(how='all')
        df['period'] = (df['date'].astype(str).str.replace('M', '-', regex=False))

        cpu_cols = [c for c in df.columns if c.startswith('cpu_index')]

        print('\n📌 CPU Columns Found:')
        print(cpu_cols)

        if len(cpu_cols) == 0:
            raise ValueError('No cpu_index column found')

        df = df[['period'] + cpu_cols]

        for i in list(df.columns):
          if i != 'period':
            df[i] = pd.to_numeric(df[i],errors='coerce') # Convert numeric safely

        df = df.dropna() # Remove NaN rows
        # Sort
        df = df.sort_values('period')
        # Reset index
        df = df.reset_index(drop=True)
        df.to_csv(save_path, index=False)

        print(
            f'\n✅ CPU downloaded successfully'
            f'\nRows: {df.shape[0]}'
            f'\nPeriod: {df["period"].min()} → {df["period"].max()}'
        )
        return df
    except Exception as e:
        print(f'\n❌ CPU download failed: {e}')
        empty = pd.DataFrame(columns=['period', 'cpu_index'])
        empty.to_csv(save_path, index=False)
        return empty

In [8]:
# ─────────────────────────────────────────────────────────────────────────────
# DOWNLOAD RAW MONTHLY CPU DATA
# ─────────────────────────────────────────────────────────────────────────────
# Phase 01B only saves the monthly source table. Rolling averages, z-scores,
# shock flags, and other derived CPU features belong in the EDA/modelling stage.

# Delete old malformed cache so we always get a fresh download
if os.path.exists(CPU_PATH):
    os.remove(CPU_PATH)

cpu_raw = download_cpu_index(CPU_URL, CPU_PATH)
print('\n📌 Final CPU monthly data:')
display(cpu_raw.head())
print(f'💾 Saved raw CPU monthly table: {CPU_PATH}')


⬇️ Downloading CPU index...

📌 Columns:
['date', 'cpu_index_narrow', 'cpu_index_broad', 'cpu_index_gavriilidis', 'cpnews_index_narrow', 'cpnews_index_broad', 'cpsent_index', 'cpu_instrument', 'cpu_shock']

📌 CPU Columns Found:
['cpu_index_narrow', 'cpu_index_broad', 'cpu_index_gavriilidis']

✅ CPU downloaded successfully
Rows: 462
Period: 1987-04 → 2025-09

📌 Final CPU monthly data:


,period,cpu_index_narrow,cpu_index_broad,cpu_index_gavriilidis
0,1987-04,33.284360,68.722160,43.589069
1,1987-05,15.797648,37.422491,41.862022
2,1987-06,69.039516,91.810961,71.135203
3,1987-07,29.499798,49.249955,53.756704
4,1987-08,53.032074,78.879270,60.615545


💾 Saved raw CPU monthly table: /content/drive/MyDrive/Regime_Detection_Swagato_M25AI2107/Data/raw/climate_policy/cpu_monthly.csv


---
## Section 6 · [NEWS-RSS] Scrape Financial News via RSS Feeds

**Why?**  
RSS feeds are the lowest-latency source of recent news (updated every few minutes).  
We use them to capture the current and trailing months, filling the gap between GDELT's ~2-day lag and today.

**Design decisions:**
- Articles older than **5 days** are skipped — RSS feeds don't retain history.
- The relevance filter (`_is_relevant`) discards off-topic articles before storage.
- BeautifulSoup strips HTML tags from summaries so FinBERT receives clean text.

**Saves:** `raw/rss/news_raw_YYYYMM.csv` (monthly slices) · `raw/rss/news_master.csv` (all periods)


In [9]:
# ─────────────────────────────────────────────────────────────────────────────
# RSS SCRAPER FUNCTION
# ─────────────────────────────────────────────────────────────────────────────

def fetch_rss_feed(name: str, url: str) -> list:
    """Fetch and filter one RSS feed, returning a list of article dicts.

    Args:
        name: Human-readable feed name (used as the 'source' column value).
        url:  RSS feed URL.

    Returns:
        List of article dicts with keys: source, published, period, title, summary, url.
    """
    articles = []
    # Only keep articles published in the last 5 days
    recent_cutoff = datetime.now() - timedelta(days=5)

    try:
        feed = feedparser.parse(url)
        print(f'[{name}] entries found: {len(feed.entries)}')

        for entry in feed.entries:
            # Parse publication timestamp — fall back to updated date if needed
            pub = None
            if hasattr(entry, 'published_parsed') and entry.published_parsed:
                pub = datetime(*entry.published_parsed[:6])
            elif hasattr(entry, 'updated_parsed') and entry.updated_parsed:
                pub = datetime(*entry.updated_parsed[:6])

            if pub is None or pub < recent_cutoff:
                continue   # Skip missing dates or articles older than 5 days

            title   = getattr(entry, 'title',   '') or ''
            summary = getattr(entry, 'summary', '') or ''
            # Strip HTML tags from the summary field
            summary = BeautifulSoup(summary, 'lxml').get_text(' ', strip=True)

            # Apply relevance filter — only keep Indian-market-relevant articles
            if not _is_relevant(f'{title}. {summary}'):
                continue

            articles.append({
                'source'   : name,
                'published': pub,
                'period'   : pub.strftime('%Y-%m'),
                'title'    : title.strip(),
                'summary'  : summary[:600],   # Cap at 600 chars for storage efficiency
                'url'      : getattr(entry, 'link', ''),
            })

    except Exception as e:
        print(f'⚠️  [{name}] feed error: {e}')

    return articles


In [10]:
# ─────────────────────────────────────────────────────────────────────────────
# RUN: Iterate all 27 RSS feeds, collect articles, save monthly slices
# ─────────────────────────────────────────────────────────────────────────────

print('\n[NEWS-RSS] Fetching all RSS feeds...\n')
all_articles = []

for name, url in tqdm(NEWS_FEEDS.items(), desc='RSS Feeds'):
    all_articles.extend(fetch_rss_feed(name, url))
    time.sleep(0.5)   # Polite crawl delay — avoids rate-limiting

news_rss = pd.DataFrame(all_articles)

if news_rss.empty:
    print('⚠️  No articles scraped — RSS feeds may be rate-limited or unreachable.')
else:
    news_rss['published'] = pd.to_datetime(news_rss['published'])
    news_rss = (
        news_rss
        .drop_duplicates(subset=['title'])   # Remove exact title duplicates
        .sort_values('published')
        .reset_index(drop=True)
    )
    print(f'\n✅ Total articles scraped : {len(news_rss):,}')
    print(f'   Period coverage        : {news_rss["period"].min()} → {news_rss["period"].max()}')
    print(f'   Source distribution    : {news_rss["source"].value_counts().to_dict()}')

    # Save one CSV per calendar month (incremental — safe to re-run)
    for period, grp in news_rss.groupby('period'):
        path = os.path.join(RSS_DIR, f'news_raw_{period.replace("-","")}.csv')
        grp.to_csv(path, index=False)
    print(f'💾 Monthly slices saved ({news_rss["period"].nunique()} months) → {RSS_DIR}/')

    # Save full master file
    master_path = os.path.join(RSS_DIR, 'news_master.csv')
    news_rss.to_csv(master_path, index=False)
    display(news_rss.head())
    print(f'💾 Master file → {master_path}')



[NEWS-RSS] Fetching all RSS feeds...



RSS Feeds:   0%|          | 0/27 [00:00<?, ?it/s]

[ET_Markets] entries found: 50
[MoneyControl_Mkt] entries found: 18
[LiveMint_Mkt] entries found: 35
[Business_Standard] entries found: 0
[CNBC_India] entries found: 0
[FinancialExpress] entries found: 0
[EconomicTimes] entries found: 76
[ET_Economy] entries found: 50
[RBI_PressReleases] entries found: 0
[SEBI_Circulars] entries found: 0
[PIB_Economy] entries found: 0
[DownToEarth] entries found: 0
[ET_Climate] entries found: 0
[Reuters_Business] entries found: 0
[Reuters_Markets] entries found: 0
[Bloomberg_Asia] entries found: 30
[Bloomberg_Economy] entries found: 30
[CNBC_World] entries found: 30
[WSJ_World] entries found: 20
[FederalReserve] entries found: 20
[ECB] entries found: 15
[IMF_News] entries found: 0
[OilPrice] entries found: 15
[Reuters_Energy] entries found: 0
[Reuters_World] entries found: 0
[BBC_World] entries found: 47
[AlJazeera] entries found: 25

✅ Total articles scraped : 177
   Period coverage        : 2026-06 → 2026-06
   Source distribution    : {'ET_Markets':

,source,published,period,title,summary,url
0,EconomicTimes,2026-06-06 00:00:00,2026-06,Central bank turns piper to draw in foreign ca...,The Reserve Bank of India has introduced measu...,https://economictimes.indiatimes.com/news/econ...
1,EconomicTimes,2026-06-06 00:30:00,2026-06,India defies West Asia war concerns as Q4 GDP ...,India's economy surged by a better-than-expect...,https://economictimes.indiatimes.com/news/econ...
2,BBC_World,2026-06-07 07:11:18,2026-06,Steve Rosenberg: Russia's economic forum overs...,The BBC's Russia editor saw Putin's flagship e...,https://www.bbc.com/news/articles/c9q2gp52rgro...
3,EconomicTimes,2026-06-07 13:36:08,2026-06,Global airlines slash 2026 profit forecast on ...,The global airline industry has significantly ...,https://economictimes.indiatimes.com/news/inte...
4,BBC_World,2026-06-08 14:11:22,2026-06,Iran's strike on Israel suggests the regime's ...,Iran's decision to risk jeopardising peace tal...,https://www.bbc.com/news/articles/cly72ryrd1jo...


💾 Master file → /content/drive/MyDrive/Regime_Detection_Swagato_M25AI2107/Data/raw/rss/news_master.csv


---
## Section 7 · [YFINANCE] Yahoo Finance News — Indian Tickers & Indices

**Why?**  
`yfinance` provides curated news directly associated with specific ticker symbols,  
giving us articles that are already pre-filtered for relevance to the exact indices we study.  
No API key required; rolling ~3–5 year coverage per ticker.

**Saves:** `raw/yfinance/yfinance_news.csv` (master cache) · `raw/yfinance/yf_news_YYYYMM.csv` (monthly)


In [11]:
def fetch_yf_news(ticker: str, from_period: str) -> list:
    """Fetch news articles from Yahoo Finance for a given ticker.

    Args:
        ticker:      yfinance ticker symbol (e.g. '^NSEI').
        from_period: 'YYYY-MM' earliest month to include.

    Returns:
        List of article dicts compatible with the unified news schema.

    Note:
        The yfinance news API wraps Yahoo Finance's internal news endpoint.
        Article structure may change without notice — the code handles both
        the legacy flat dict and the newer nested 'content' dict structure.
    """
    from_dt = pd.Period(from_period, 'M').to_timestamp()
    articles = []
    try:
        t = yf.Ticker(ticker)
        news_items = t.news or []
        for item in news_items:
            ts = item.get('providerPublishTime')
            if not ts:
                continue
            pub = pd.to_datetime(ts, unit='s', utc=True).tz_localize(None)
            if pub < from_dt:
                continue
            # Handle both flat and nested content schemas
            content   = item.get('content', {})
            title     = content.get('title', '') or item.get('title', '') or ''
            summary   = content.get('summary', '') or item.get('summary', '') or ''
            url       = (content.get('canonicalUrl', {}).get('url', '')
                         or item.get('link', ''))
            publisher = (content.get('provider', {}).get('displayName', '')
                         or item.get('publisher', 'YahooFinance'))
            if not title:
                continue
            articles.append({
                'source'   : f'YFinance:{ticker}',
                'published': pub,
                'period'   : pub.strftime('%Y-%m'),
                'title'    : title.strip()[:400],
                'summary'  : str(summary).strip()[:600],
                'url'      : url,
                'publisher': publisher,
            })
    except Exception as e:
        print(f'   ⚠️  [{ticker}] yfinance error: {e}')
    return articles


In [12]:
# ─────────────────────────────────────────────────────────────────────────────
# RUN: Load from cache (if available) or fetch fresh from yfinance
# ─────────────────────────────────────────────────────────────────────────────
# Caching avoids redundant API calls on re-runs and prevents daily rate limits.

YF_MASTER = os.path.join(YF_DIR, 'yfinance_news.csv')

if os.path.exists(YF_MASTER):
    # Cache hit — load existing data
    yf_all = pd.read_csv(YF_MASTER)
    yf_all['published'] = pd.to_datetime(yf_all['published'], errors='coerce')
    print(f'✅ Loaded yfinance cache: {len(yf_all):,} articles')
else:
    # Cache miss — fetch from yfinance for all tickers
    yf_articles = []
    for ticker in tqdm(YF_TICKERS, desc='yfinance tickers'):
        yf_articles.extend(fetch_yf_news(ticker, YF_SCRAPE_FROM))
        time.sleep(0.8)

    yf_all = pd.DataFrame(yf_articles)
    if not yf_all.empty:
        yf_all['published'] = pd.to_datetime(yf_all['published'], errors='coerce')
        yf_all = (
            yf_all
            .drop_duplicates(subset=['title'])
            .sort_values('published')
            .reset_index(drop=True)
        )
        yf_all.to_csv(YF_MASTER, index=False)
        print(f'✅ yfinance: {len(yf_all):,} articles | '
              f'{yf_all["period"].min()} → {yf_all["period"].max()}')
        print(f'   Top publishers:\n{yf_all["publisher"].value_counts().head(8).to_string()}')
    else:
        yf_all = pd.DataFrame()
        print('⚠️  yfinance returned no articles.')

# Save monthly slices for alignment with other sources
if not yf_all.empty:
    for period, grp in yf_all.groupby('period'):
        path = os.path.join(YF_DIR, f'yf_news_{period.replace("-","")}.csv')
        grp.to_csv(path, index=False)
    print(f'💾 Monthly yfinance slices → {YF_DIR}/')


yfinance tickers:   0%|          | 0/6 [00:00<?, ?it/s]

⚠️  yfinance returned no articles.


---
## Section 8 · [GDELT-2.0] GDELT 2.0 Doc API — Feb 2015 → Mar 2026

**Why?**  
GDELT monitors virtually the entire global news media in real time.  
The 2.0 Doc API provides full-text article records with URLs, tones, and themes.  
It is our **primary source** for the post-2015 period (~136 monthly windows).

**Design decisions:**
- One **calendar-month window** per API call (GDELT's 250-record limit applies per request).
- Each month is saved as a separate CSV (`gdelt2_YYYY-MM.csv`) — if a month already exists on disk, it is skipped entirely, making the loop **incremental and crash-safe**.
- `tag_keywords()` labels each article with which GDELT queries matched its title, enabling keyword-level analysis later.

**Source:** https://www.gdeltproject.org/  
**Saves:** `raw/gdelt/gdelt2_YYYY-MM.csv` (per-month) · `raw/gdelt/gdelt2_master.csv` (full merge)


In [13]:
# ─────────────────────────────────────────────────────────────────────────────
# GDELT 2.0 HELPER FUNCTIONS
# ─────────────────────────────────────────────────────────────────────────────

def generate_date_windows(start: date, end: date) -> list:
    """Generate a list of (start_str, end_str) tuples — one per calendar month.

    GDELT 2.0 allows at most 250 records per request, so we split the full
    range into monthly windows rather than querying the entire date range at once.
    """
    windows = []
    cur = start.replace(day=1)
    while cur <= end:
        next_month = (cur.replace(day=28) + timedelta(days=4)).replace(day=1)
        w_end = min(next_month - timedelta(days=1), end)
        windows.append((cur.strftime('%Y-%m-%d'), w_end.strftime('%Y-%m-%d')))
        cur = next_month
    return windows


def fetch_gdelt2_articles(start_str, end_str, gd, retries=3):
    """Query GDELT 2.0 for articles mentioning 'India' in a date window.

    Uses exponential back-off on failure (1s, 2s, 4s) before giving up.
    Returns an empty DataFrame if all retries fail.
    """
    for attempt in range(retries):
        try:
            f = Filters(
                keyword='India',
                start_date=start_str,
                end_date=end_str,
                num_records=250,
            )
            df = gd.article_search(f)
            if df is not None and not df.empty:
                return df
        except Exception:
            if attempt < retries - 1:
                time.sleep(2 ** attempt)
    return pd.DataFrame()


def tag_keywords(text):
    """Label an article title with all matching GDELT query strings.

    Used later to filter/group articles by topic (e.g. 'India ESG' articles).
    Multiple matches are pipe-delimited.
    """
    text = str(text).lower()
    matched = [kw for kw in GDELT_QUERIES if kw.lower() in text]
    return '|'.join(matched) if matched else 'general'


def clean_gdelt2_df(df):
    """Normalise raw GDELT 2.0 output to the unified news schema.

    GDELT's 'seendate' field uses the format 'YYYYMMDDTHHMMSSz'.
    We parse it, derive 'period' (YYYY-MM), and tag keywords.
    """
    df = df.copy()
    df['published'] = pd.to_datetime(
        df['seendate'], format='%Y%m%dT%H%M%SZ', utc=True, errors='coerce'
    ).dt.tz_localize(None)
    df = df.dropna(subset=['published'])
    df['period']   = df['published'].dt.strftime('%Y-%m')
    df['title']    = df.get('title', '').fillna('')
    df['url']      = df.get('url',   '').fillna('')
    df['keywords'] = df['title'].apply(tag_keywords)
    return pd.DataFrame({
        'source'   : 'GDELT2',
        'published': df['published'],
        'period'   : df['period'],
        'title'    : df['title'],
        'summary'  : '',   # GDELT 2.0 does not provide summaries
        'url'      : df['url'],
        'keywords' : df['keywords'],
    })


def process_gdelt2_window(window):
    """Fetch, clean, and cache one monthly window (skip if already on disk).

    This function is safe to call in parallel via ThreadPoolExecutor.
    Returns None if the cache already exists or no data was returned.
    """
    w_start, w_end = window
    month_str  = w_start[:7]
    cache_path = os.path.join(GDELT_DIR, f'gdelt2_{month_str}.csv')
    if os.path.exists(cache_path):
        return None   # Already downloaded — skip
    raw = fetch_gdelt2_articles(w_start, w_end, gd)
    if raw.empty:
        return None
    cleaned = clean_gdelt2_df(raw)
    cleaned.to_csv(cache_path, index=False)
    return cleaned

print('✅ GDELT 2.0 helper functions defined.')


✅ GDELT 2.0 helper functions defined.


In [14]:
# ─────────────────────────────────────────────────────────────────────────────
# RUN: Parallel GDELT 2.0 fetch across all monthly windows
# ─────────────────────────────────────────────────────────────────────────────
# We use a ThreadPoolExecutor so multiple months are fetched concurrently.
# The number of threads is capped at MAX_WORKERS (=8) to avoid hammering GDELT.

gd      = GdeltDoc()
windows = generate_date_windows(GDELT2_START, GDELT2_END)
print(f'📊 Total monthly windows to process: {len(windows)}')

gdelt2_frames = []
print('⚡ Starting parallel GDELT 2.0 fetch...\n')

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = [executor.submit(process_gdelt2_window, w) for w in windows]
    for f in tqdm(as_completed(futures), total=len(futures), desc='GDELT 2.0'):
        res = f.result()
        if res is not None:
            gdelt2_frames.append(res)

# Also load any previously cached monthly files not returned by the parallel run
cached_files = sorted(Path(GDELT_DIR).glob('gdelt2_*.csv'))
print(f'Cached monthly files on disk: {len(cached_files)}')
already_loaded = {df['period'].iloc[0] for df in gdelt2_frames if not df.empty}
for f in cached_files:
    month = f.stem.replace('gdelt2_', '')
    if month not in already_loaded:
        gdelt2_frames.append(pd.read_csv(f))

# Merge all months into a single master file
if gdelt2_frames:
    gdelt2_all = (
        pd.concat(gdelt2_frames, ignore_index=True)
        .drop_duplicates(subset=['title'])
        .sort_values('published')
        .reset_index(drop=True)
    )
    master_path = os.path.join(GDELT_DIR, 'gdelt2_master.csv')
    display(gdelt2_all.head())
    gdelt2_all.to_csv(master_path, index=False)
    print(f'\n✅ GDELT 2.0 total articles: {len(gdelt2_all):,}')
    print(f'💾 Master file → {master_path}')
else:
    gdelt2_all = pd.DataFrame()
    print('⚠️  No GDELT 2.0 data fetched.')


📊 Total monthly windows to process: 134
⚡ Starting parallel GDELT 2.0 fetch...



GDELT 2.0:   0%|          | 0/134 [00:00<?, ?it/s]

Cached monthly files on disk: 112


,source,published,period,title,summary,url,keywords
0,GDELT2,2017-01-08 02:00:00,2017-01,"Editorial column , 08",NaN,https://jang.com.pk/print/246067-editorial-col...,general
1,GDELT2,2017-01-08 04:00:00,2017-01,ভারতের ক্ষেপনাস্ত্র পরীক্ষায় কি অস্ত্র প্রতিযো...,NaN,http://www.thebengalitimes.com/international/2...,general
2,GDELT2,2017-01-08 04:30:00,2017-01,為什麼蘋果手機在印度無人問津 ？ 印度人 ： 我們又不傻,NaN,http://news.sina.com.tw/article/20170108/20080...,general
3,GDELT2,2017-01-08 05:00:00,2017-01,征戰清奈 盧彥勳胞兄 ： 辦印度簽證屢遭刁難,NaN,http://www.chinatimes.com/realtimenews/2017010...,general
4,GDELT2,2017-01-08 05:15:00,2017-01,中国厂商真的在 杀印度手机品牌吗 ？- 凤凰国际智库,NaN,http://pit.ifeng.com/a/20170108/50540364_0.shtml,general



✅ GDELT 2.0 total articles: 25,816
💾 Master file → /content/drive/MyDrive/Regime_Detection_Swagato_M25AI2107/Data/raw/gdelt/gdelt2_master.csv


---
## Section 9 · [GDELT-1.0] GDELT 1.0 Raw CSV — 2000 → Feb 2015

**Why?**  
GDELT 2.0 only covers from Feb 2015. To extend sentiment coverage back to 2000  
(the full NIFTY 50 listed history), we use GDELT 1.0 raw daily event CSV files.

**What we extract:**  
GDELT 1.0 records bilateral events between countries. We filter for events  
where India is `country1` or `country2`, then compute a **weighted monthly AvgTone**  
(weighted by `mentions` count). AvgTone is normalised to [−1, +1] to match FinBERT scale.

> `gdelt1_tone_norm` is used as a proxy net sentiment for pre-2015 months where  
> no FinBERT score exists.

**Source:** http://data.gdeltproject.org/  
**Saves:** `raw/gdelt/gdelt1_YYYY-MM.csv` (monthly raw) · `raw/gdelt/gdelt1_monthly_tone.csv` (features)


In [15]:
# ─────────────────────────────────────────────────────────────────────────────
# GDELT 1.0 HELPER FUNCTIONS
# ─────────────────────────────────────────────────────────────────────────────

# GDELT 1.0 daily export columns we actually need (saves bandwidth vs loading all ~60 cols)
GDELT1_COLS = [1, 7, 17, 30, 31, 34]
COL_NAMES   = ['day', 'country1', 'country2', 'goldstein', 'mentions', 'avg_tone']


def fetch_day(year, month, day):
    """Download and filter one day's GDELT 1.0 export CSV.

    Each daily file is a zipped TSV (~10-80 MB uncompressed). We stream
    only the 6 required columns and immediately filter for India events.
    Returns a small DataFrame or None if the file is unavailable.
    """
    date_str = f'{year}{month:02d}{day:02d}'
    url = f'http://data.gdeltproject.org/events/{date_str}.export.CSV.zip'

    try:
        r = requests.get(url, headers=HEADERS, timeout=30)
        if r.status_code != 200:
            return None   # File may not exist for this date

        with zipfile.ZipFile(io.BytesIO(r.content)) as z:
            with z.open(z.namelist()[0]) as f:
                df = pd.read_csv(
                    f, sep='\t', header=None,
                    usecols=GDELT1_COLS, names=COL_NAMES, low_memory=False
                )

        # Keep only events involving India as actor1 OR actor2
        df = df[(df['country1'] == 'IND') | (df['country2'] == 'IND')]
        if df.empty:
            return None

        # Coerce numeric columns (some cells contain '#VALUE!' or similar)
        df['mentions']  = pd.to_numeric(df['mentions'],  errors='coerce').fillna(1)
        df['avg_tone']  = pd.to_numeric(df['avg_tone'],  errors='coerce')
        df['goldstein'] = pd.to_numeric(df['goldstein'], errors='coerce')
        df = df.dropna(subset=['avg_tone'])

        return df[['day', 'avg_tone', 'goldstein', 'mentions']]

    except Exception as e:
        print(f'❌ {date_str} error: {e}')
        return None


def process_month(year, month):
    """Fetch all days in a month in parallel and save the raw monthly CSV.

    Returns a DataFrame with all India GDELT events for the month,
    or None if no data was retrieved.
    """
    print(f'\n📦 Processing {year}-{month:02d}')
    _, days = calendar.monthrange(year, month)

    results = []
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = [executor.submit(fetch_day, year, month, d) for d in range(1, days + 1)]
        for f in futures:
            res = f.result()
            if res is not None:
                results.append(res)

    if not results:
        print(f'⚠️  No data for {year}-{month:02d}')
        return None

    df_month = pd.concat(results, ignore_index=True)
    df_month['period'] = f'{year}-{month:02d}'

    out_file = os.path.join(GDELT_DIR, f'gdelt1_{year}-{month:02d}.csv')
    df_month.to_csv(out_file, index=False)
    print(f'   ✅ Saved {out_file} | rows={len(df_month):,}')
    return df_month


In [16]:
# ─────────────────────────────────────────────────────────────────────────────
# RUN: Iterate 2000-01 → 2015-01, one month at a time
# ─────────────────────────────────────────────────────────────────────────────
# NOTE: This loop downloads ~180 months of GDELT 1.0 data.
# Each month takes ~30s on Colab. Expect 60-90 minutes total runtime.
# The loop is resumable — each month is saved immediately so Colab crashes
# lose at most one month of progress.

all_months = []
cur = GDELT1_START

while (cur.year < GDELT1_END.year) or (cur.year == GDELT1_END.year and cur.month <= GDELT1_END.month):
    # Skip months that are already cached on disk
    cache_file = os.path.join(GDELT_DIR, f'gdelt1_{cur.year}-{cur.month:02d}.csv')
    if os.path.exists(cache_file):
        df_cached = pd.read_csv(cache_file)
        if not df_cached.empty:
            all_months.append(df_cached)
    else:
        df = process_month(cur.year, cur.month)
        if df is not None:
            all_months.append(df)
    # Advance to next month
    if cur.month == 12:
        cur = datetime(cur.year + 1, 1, 1)
    else:
        cur = datetime(cur.year, cur.month + 1, 1)

if not all_months:
    print('⚠️  No GDELT 1.0 data retrieved — check network access.')
    gdelt1_monthly = pd.DataFrame(columns=['period', 'gdelt1_tone_raw', 'gdelt1_tone_norm',
                                            'gdelt1_goldstein', 'gdelt1_articles'])
else:
    # ── Save full raw combined file ────────────────────────────────────────────────
    full_df   = pd.concat(all_months, ignore_index=True)
    full_path = os.path.join(GDELT_DIR, 'gdelt1_full.csv')
    full_df.to_csv(full_path, index=False)
    print(f'\n✅ Combined raw saved: {full_path}')

    # ── Compute monthly features ───────────────────────────────────────────────────
    # Weighted average tone: events with more media mentions carry more weight.
    gdelt1_monthly = full_df.groupby('period').apply(
        lambda g: pd.Series({
            'gdelt1_tone_raw'  : np.average(g['avg_tone'], weights=g['mentions']),
            'gdelt1_goldstein' : g['goldstein'].mean(),
            'gdelt1_articles'  : len(g),
        })
    ).reset_index()

    # Normalise tone from raw range to [−1, +1] to match FinBERT sentiment scale
    tone_min = gdelt1_monthly['gdelt1_tone_raw'].min()
    tone_max = gdelt1_monthly['gdelt1_tone_raw'].max()
    gdelt1_monthly['gdelt1_tone_norm'] = (
        2 * (gdelt1_monthly['gdelt1_tone_raw'] - tone_min) / (tone_max - tone_min + 1e-9) - 1
    )

    feature_path = os.path.join(GDELT_DIR, 'gdelt1_monthly_tone.csv')
    gdelt1_monthly.to_csv(feature_path, index=False)
    print(f'✅ Monthly features saved: {feature_path}')
    display(gdelt1_monthly.head())



📦 Processing 2000-01
⚠️  No data for 2000-01

📦 Processing 2000-02
⚠️  No data for 2000-02

📦 Processing 2000-03
⚠️  No data for 2000-03

📦 Processing 2000-04
⚠️  No data for 2000-04

📦 Processing 2000-05
⚠️  No data for 2000-05

📦 Processing 2000-06
⚠️  No data for 2000-06

📦 Processing 2000-07
⚠️  No data for 2000-07

📦 Processing 2000-08
⚠️  No data for 2000-08

📦 Processing 2000-09
⚠️  No data for 2000-09

📦 Processing 2000-10
⚠️  No data for 2000-10

📦 Processing 2000-11
⚠️  No data for 2000-11

📦 Processing 2000-12
⚠️  No data for 2000-12

📦 Processing 2001-01
⚠️  No data for 2001-01

📦 Processing 2001-02
⚠️  No data for 2001-02

📦 Processing 2001-03
⚠️  No data for 2001-03

📦 Processing 2001-04
⚠️  No data for 2001-04

📦 Processing 2001-05
⚠️  No data for 2001-05

📦 Processing 2001-06
⚠️  No data for 2001-06

📦 Processing 2001-07
⚠️  No data for 2001-07

📦 Processing 2001-08
⚠️  No data for 2001-08

📦 Processing 2001-09
⚠️  No data for 2001-09

📦 Processing 2001-10
⚠️  No data 

,period,gdelt1_tone_raw,gdelt1_goldstein,gdelt1_articles,gdelt1_tone_norm
0,2013-04,2.006373,-0.412384,15787.0,-1.000000
1,2013-05,2.125246,0.188622,15108.0,-0.670042
2,2013-06,2.384107,0.684164,27299.0,0.048482
3,2013-07,2.187072,0.190834,45081.0,-0.498432
4,2013-08,2.055116,-0.400583,56087.0,-0.864706


---
## Section 10 · [ALPHAVANTAGE] Alpha Vantage News Sentiment — Recent ~2 Years

**Why?**  
Alpha Vantage provides **pre-scored** news sentiment (bullish/bearish/neutral probability scores)  
from a curated financial news corpus. This gives us a second sentiment signal independent of  
FinBERT that can be used to validate or ensemble sentiment estimates.

**Limitations:**
- Free tier: 25 requests/day, 200 articles per request.
- Rolling ~2-year window (older articles are dropped from the API).
- We cache each topic to a CSV so daily limits aren't wasted on re-runs.

**Saves:** `raw/alphavantage/av_{topic}.csv` · `raw/alphavantage/av_monthly_sentiment.csv`


In [17]:
AV_BASE         = 'https://www.alphavantage.co/query'
AV_MONTHLY_PATH = os.path.join(AV_DIR, 'av_monthly_sentiment.csv')

def fetch_av_news(topic: str, api_key: str, retries: int = 3) -> pd.DataFrame:
    """Fetch pre-scored news sentiment from Alpha Vantage for one topic bucket.

    Args:
        topic:   AV topic string (e.g. 'financial_markets').
        api_key: Alpha Vantage API key.
        retries: Number of attempts before giving up.

    Returns:
        DataFrame with columns: source, published, period, title, summary,
        url, av_label, av_score.
    """
    params = {
        'function': 'NEWS_SENTIMENT',
        'topics'  : topic,
        'sort'    : 'LATEST',
        'limit'   : 200,
        'apikey'  : api_key,
    }
    for attempt in range(retries):
        try:
            r = requests.get(AV_BASE, params=params, timeout=30)
            r.raise_for_status()
            data = r.json()
            if 'feed' not in data:
                return pd.DataFrame()
            rows = []
            for item in data['feed']:
                pub = pd.to_datetime(
                    item.get('time_published', ''),
                    format='%Y%m%dT%H%M%S', errors='coerce'
                )
                if pd.isna(pub):
                    continue
                rows.append({
                    'source'   : 'AlphaVantage',
                    'published': pub,
                    'period'   : pub.strftime('%Y-%m'),
                    'title'    : item.get('title', ''),
                    'summary'  : item.get('summary', '')[:600],
                    'url'      : item.get('url', ''),
                    'av_label' : item.get('overall_sentiment_label', ''),
                    'av_score' : item.get('overall_sentiment_score', None),
                })
            return pd.DataFrame(rows)
        except Exception as e:
            if attempt < retries - 1:
                time.sleep(2 ** attempt)   # Exponential back-off
            else:
                print(f'⚠️  Failed topic={topic}: {e}')
    return pd.DataFrame()


In [18]:
# ─────────────────────────────────────────────────────────────────────────────
# RUN: Fetch all AV topic buckets (with caching and daily limit guard)
# ─────────────────────────────────────────────────────────────────────────────
av_frames  = []
call_count = 0
MAX_CALLS  = 20   # Safety cap — free tier allows 25/day; leave headroom

if AV_API_KEY:
    for topic in tqdm(AV_TOPICS, desc='Alpha Vantage topics'):
        if call_count >= MAX_CALLS:
            print('⚠️  Daily API limit reached. Stopping early.')
            break
        cache_path = os.path.join(AV_DIR, f'av_{topic}.csv')
        if os.path.exists(cache_path):
            av_frames.append(pd.read_csv(cache_path))   # Load from cache
            continue
        df = fetch_av_news(topic, AV_API_KEY)
        if not df.empty:
            df.to_csv(cache_path, index=False)
            av_frames.append(df)
        call_count += 1
        time.sleep(15)   # AV free tier: max ~4 requests/minute
    print(f'\n✅ Raw articles fetched: {sum(len(d) for d in av_frames):,}')
else:
    print('⚠️  AV_API_KEY not set. Add ALPHA_VANTAGE_API_KEY to Colab Secrets to fetch Alpha Vantage data.')

av_all = pd.concat(av_frames, ignore_index=True) if av_frames else pd.DataFrame()

if not av_all.empty:
    av_all['published'] = pd.to_datetime(av_all['published'])
    av_all['av_score']  = pd.to_numeric(av_all['av_score'], errors='coerce')
    av_all = av_all.drop_duplicates(subset=['title', 'published'])
    display(av_all.head())
    print(f'🧹 After deduplication: {len(av_all):,} articles')

    # Aggregate to monthly statistics for the COMBINE step
    av_monthly = (
        av_all.groupby('period')
        .agg(
            av_sentiment_mean=('av_score', 'mean'),
            av_sentiment_std =('av_score', 'std'),
            av_article_count =('av_score', 'count'),
        )
        .reset_index()
        .sort_values('period')
    )
    av_monthly.to_csv(AV_MONTHLY_PATH, index=False)
    print(f'💾 Monthly AV sentiment → {AV_MONTHLY_PATH}')
    display(av_monthly.tail())
else:
    print('⚠️  No Alpha Vantage data available.')


Alpha Vantage topics:   0%|          | 0/8 [00:00<?, ?it/s]


✅ Raw articles fetched: 400


,source,published,period,title,summary,url,av_label,av_score
0,AlphaVantage,2026-04-29 06:12:10,2026-04,PACCAR Inc (PCAR) Q1 2026 Earnings Call Highli...,PACCAR Inc (PCAR) reported robust first-quarte...,https://www.gurufocus.com/news/8825862/paccar-...,Bullish,0.400145
1,AlphaVantage,2026-04-29 06:11:38,2026-04,Xylem Inc (XYL) Q1 2026 Earnings Call Highligh...,Xylem Inc (XYL) reported resilient Q1 2026 fin...,https://www.gurufocus.com/news/8825833/xylem-i...,Somewhat-Bullish,0.290782
2,AlphaVantage,2026-04-29 06:00:00,2026-04,China suspends new autonomous vehicle permits ...,China has reportedly suspended the issuance of...,https://www.reuters.com/world/asia-pacific/chi...,Bearish,-0.723555
3,AlphaVantage,2026-04-29 05:25:22,2026-04,Gold Shrugs Off the Oil Crisis – Who Are the W...,Despite escalating Middle East conflicts and r...,https://news.financial/comments/gold-shrugs-of...,Bullish,0.389099
4,AlphaVantage,2026-04-29 05:18:11,2026-04,NRG Energy Debt Tender Reshapes Covenants And ...,"NRG Energy's subsidiary, Lightning Power, has ...",https://simplywall.st/stocks/us/utilities/nyse...,Somewhat-Bullish,0.180500


🧹 After deduplication: 305 articles
💾 Monthly AV sentiment → /content/drive/MyDrive/Regime_Detection_Swagato_M25AI2107/Data/raw/alphavantage/av_monthly_sentiment.csv


,period,av_sentiment_mean,av_sentiment_std,av_article_count
0,2026-04,0.112940,0.284242,212
1,2026-05,0.235213,0.233717,93


---
## Section 11 · [WAYBACK] Wayback Machine CDX API — Indian News Archives 2006–2015

**Why?**  
Neither RSS feeds nor GDELT 2.0 cover the pre-2015 period well for Indian financial news.  
The Wayback Machine's **CDX API** lets us enumerate archived URLs for major Indian news domains  
and then fetch article titles from the Wayback HTML pages.

**Two-step process:**
1. **CDX query** — enumerate all archived URLs under `/markets/`, `/economy/`, `/news/`, `/finance/`.
2. **Title fetch** — load each archived page and extract the `<title>` tag.

**Coverage:** `economictimes.indiatimes.com` (2008+), `livemint.com` (2009+),  
`business-standard.com` (2007+), `moneycontrol.com` (2008+), `thehindubusinessline.com` (2006+)

**Saves:** `raw/wayback/wayback_titles.csv` (incremental — safe to restart)


In [19]:
# ─────────────────────────────────────────────────────────────────────────────
# WAYBACK MACHINE HELPER FUNCTIONS
# ─────────────────────────────────────────────────────────────────────────────

def fetch_wayback_cdx(domain, from_year=2008, to_year=2015):
    """Query the CDX API for all archived URLs of a domain within a date range.

    The CDX API returns a JSON array where row 0 is the header.
    We iterate paths (/markets/, /economy/, etc.) separately to avoid
    hitting CDX record limits.

    Returns:
        DataFrame with columns: published, period, url, url_live.
    """
    news_paths = ['/markets/', '/economy/', '/news/', '/finance/']
    CDX_URL    = 'http://web.archive.org/cdx/search/cdx'
    all_rows   = []

    for path in news_paths:
        params = {
            'url'    : f'{domain}{path}*',
            'output' : 'json',
            'fl'     : 'timestamp,original,statuscode',
            'filter' : 'statuscode:200',
            'from'   : f'{from_year}0101',
            'to'     : f'{to_year}1231',
            'limit'  : 20000,
            'collapse': 'urlkey',   # Deduplicate by URL key (ignores query strings)
        }
        try:
            r    = requests.get(CDX_URL, params=params, headers=HEADERS, timeout=60)
            data = r.json()
            if len(data) <= 1:
                continue   # Only header row — no results for this path
            for row in data[1:]:
                ts, url, _ = row
                try:
                    dt = pd.to_datetime(ts, format='%Y%m%d%H%M%S')
                    all_rows.append({
                        'published': dt,
                        'period'   : dt.strftime('%Y-%m'),
                        'url'      : url,
                        'url_live' : f'https://web.archive.org/web/{ts}/{url}',
                    })
                except Exception:
                    continue
        except Exception as e:
            print(f'⚠️  CDX error for {domain}{path}: {e}')
        time.sleep(0.3)

    return pd.DataFrame(all_rows)


def fetch_wayback_title(url, session):
    """Fetch a Wayback Machine page and extract the <title> tag.

    Short titles (< 20 chars) or error pages ('404' in title) are discarded.
    Retries up to 3 times with exponential back-off.
    """
    for attempt in range(3):
        try:
            r         = session.get(url, headers=HEADERS, timeout=15)
            soup      = BeautifulSoup(r.text, 'lxml')
            title_tag = soup.find('title')
            if not title_tag:
                return ''
            title = title_tag.get_text(strip=True)
            if len(title) < 20 or '404' in title.lower():
                return ''
            return title
        except Exception:
            time.sleep(2 ** attempt)
    return ''


def fetch_titles_parallel(sample_df, domain):
    """Fetch article titles for a sample of CDX-enumerated URLs in parallel.

    Applies the relevance filter so only India-market-relevant titles are kept.
    """
    results = []
    session = requests.Session()

    def task(row):
        title = fetch_wayback_title(row['url_live'], session)
        if not title or not _is_relevant(title):
            return None
        return {
            'source'   : f'Wayback:{domain.split(".")[0]}',
            'domain'   : domain,
            'published': row['published'],
            'period'   : row['period'],
            'title'    : title[:300],
            'summary'  : '',   # No summary available from title-only scraping
            'url'      : row['url_live'],
        }

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = [executor.submit(task, row) for _, row in sample_df.iterrows()]
        for f in as_completed(futures):
            res = f.result()
            if res:
                results.append(res)

    return results

print('✅ Wayback CDX helper functions defined.')

✅ Wayback CDX helper functions defined.


In [20]:
# ─────────────────────────────────────────────────────────────────────────────
# RUN: CDX enumeration → title scraping (incremental, per-domain-per-month)
# ─────────────────────────────────────────────────────────────────────────────
WB_CACHE = os.path.join(WAYBACK_DIR, 'wayback_titles.csv')

# Load existing cache so we can skip already-processed (domain, period) pairs
if os.path.exists(WB_CACHE):
    wb_existing = pd.read_csv(WB_CACHE)
    wb_existing['published'] = pd.to_datetime(wb_existing['published'], errors='coerce')
    wb_done = set(zip(wb_existing['domain'], wb_existing['period']))
    print(f'✅ Loaded Wayback cache: {len(wb_existing):,} articles')
else:
    wb_existing = pd.DataFrame()
    wb_done     = set()

wb_frames = [wb_existing] if not wb_existing.empty else []

for domain, from_year in WAYBACK_DOMAINS.items():
    print(f'\n📚 Processing: {domain} ({from_year}–2015)')

    cdx = fetch_wayback_cdx(domain, from_year, 2015)
    if cdx.empty:
        print('   No CDX results found for this domain.')
        continue

    for period, grp in cdx.groupby('period'):
        if (domain, period) in wb_done:
            continue   # Already processed — skip

        # Sample the most-recent N articles in the month (avoids scraping thousands of URLs)
        sample = grp.sort_values('published').tail(min(WAYBACK_SAMPLE_PER_MONTH, len(grp)))
        titles = fetch_titles_parallel(sample, domain)

        if titles:
            chunk = pd.DataFrame(titles)
            wb_frames.append(chunk)
            wb_done.add((domain, period))
            # Incremental save after each (domain, period) — crash-safe
            pd.concat(wb_frames, ignore_index=True).to_csv(WB_CACHE, index=False)
            print(f'   {period}: {len(titles)} relevant titles saved')

wb_all = pd.concat(wb_frames, ignore_index=True) if wb_frames else pd.DataFrame()
if not wb_all.empty:
    wb_all = wb_all.drop_duplicates(subset=['url'])

print(f'\n✅ Total Wayback articles: {len(wb_all):,}')
print(f'💾 Saved: {WB_CACHE}')
display(wb_all)


✅ Loaded Wayback cache: 2,085 articles

✅ Total Wayback articles: 2,071
💾 Saved: /content/drive/MyDrive/Regime_Detection_Swagato_M25AI2107/Data/raw/wayback/wayback_titles.csv


,source,domain,published,period,title,summary,url
0,Wayback:economictimes,economictimes.indiatimes.com,2008-01-02 03:27:57,2008-01,Five regional scheduled airline operators to g...,NaN,https://web.archive.org/web/20080102032757/htt...
1,Wayback:economictimes,economictimes.indiatimes.com,2008-01-02 13:10:16,2008-01,Threat to food security: Grain crisis- Agricul...,NaN,https://web.archive.org/web/20080102131016/htt...
2,Wayback:economictimes,economictimes.indiatimes.com,2008-01-14 02:25:35,2008-01,Forex reserves rise to $273 bn- Forex-Markets-...,NaN,https://web.archive.org/web/20080114022535/htt...
3,Wayback:economictimes,economictimes.indiatimes.com,2008-01-01 12:28:24,2008-01,Banks' profit growth trails income- Finance-Ec...,NaN,https://web.archive.org/web/20080101122824/htt...
4,Wayback:economictimes,economictimes.indiatimes.com,2008-01-21 00:18:14,2008-01,Global credit market turmoil may abate by '08-...,NaN,https://web.archive.org/web/20080121001814/htt...
...,...,...,...,...,...,...,...
2080,Wayback:thehindubusinessline,thehindubusinessline.com,2013-06-30 07:10:58,2013-06,Weak rupee cushions fall in imported edible oi...,NaN,https://web.archive.org/web/20130630071058/htt...
2081,Wayback:thehindubusinessline,thehindubusinessline.com,2013-06-30 20:10:27,2013-06,"Crude oil futures jump to Rs 5,832 per barrel ...",NaN,https://web.archive.org/web/20130630201027/htt...
2082,Wayback:thehindubusinessline,thehindubusinessline.com,2013-06-30 15:35:38,2013-06,Coal India shares hit all-time low | Business ...,NaN,https://web.archive.org/web/20130630153538/htt...
2083,Wayback:thehindubusinessline,thehindubusinessline.com,2013-06-30 22:24:02,2013-06,Coal India shares hit all-time low | Business ...,NaN,https://web.archive.org/web/20130630222402/htt...


---
## Section 12 · [MERGE-RAW] Combine All Sources → Unified News Corpus

**Why?**  
Before running sentiment, we unify all five sources into a single consistent schema.  
Deduplication is critical: the same article often appears in GDELT, RSS, and yfinance simultaneously.

**Deduplication strategy (two passes):**
1. Exact URL match.
2. Normalised title match (lower-case, strip punctuation/URLs, collapse whitespace).

**Source quality weights** (`source_weight` column) allow downstream models to  
down-weight noisier sources (e.g. Wayback title-only scrapes = 0.6) relative to  
structured feeds (AlphaVantage = 1.2).

**Saves:** `processed/news_corpus_all.csv`


In [21]:
# ─────────────────────────────────────────────────────────────────────────────
# MERGE-RAW HELPER FUNCTIONS
# ─────────────────────────────────────────────────────────────────────────────

def normalize_title(t: str) -> str:
    """Produce a canonical lower-case, punctuation-free title for deduplication.

    Two articles are treated as duplicates if their normalised titles match,
    even if formatting or capitalisation differs.
    """
    if pd.isna(t):
        return ''
    t = t.lower()
    t = re.sub(r'http\S+', ' ', t)        # Remove embedded URLs
    t = re.sub(r'[^a-z0-9 ]', ' ', t)     # Remove punctuation
    t = re.sub(r'\s+', ' ', t).strip()    # Collapse whitespace
    return t


def standardize_df(df, label: str) -> pd.DataFrame:
    """Enforce a consistent 8-column schema across all source DataFrames.

    Columns guaranteed in output:
        source, published, period, title, summary, url, av_label, av_score

    Missing columns are filled with '' or None as appropriate.
    Short titles (< 20 chars) are dropped — they are almost always noise.
    """
    if df is None or not isinstance(df, pd.DataFrame) or df.empty:
        return pd.DataFrame()
    df = df.copy()
    for col in ['title', 'summary', 'url', 'source']:
        if col not in df.columns:
            df[col] = ''
    for col in ['av_label', 'av_score']:
        if col not in df.columns:
            df[col] = None
    df['published'] = pd.to_datetime(df['published'], errors='coerce')
    df = df.dropna(subset=['published'])
    df['period']  = df['published'].dt.strftime('%Y-%m')
    df['title']   = df['title'].astype(str).str.strip()
    df['summary'] = df['summary'].astype(str).str.strip()
    df['url']     = df['url'].astype(str).str.strip()
    df['source']  = df['source'].astype(str)
    df.loc[df['source'].eq('') | df['source'].isna(), 'source'] = label
    df = df[['source', 'published', 'period', 'title', 'summary', 'url', 'av_label', 'av_score']]
    print(f'{label:12s}: {len(df):7,} articles | {df["period"].min()} → {df["period"].max()}')
    return df

In [22]:
# ─────────────────────────────────────────────────────────────────────────────
# RUN: Load all sources, standardise, merge, deduplicate
# ─────────────────────────────────────────────────────────────────────────────

# Load each source from disk (safe even if in-memory variable is stale)
gdelt2_files = sorted(f for f in Path(GDELT_DIR).glob('gdelt2_*.csv')
                if f.stem != 'gdelt2_master')  # exclude the merged master file
gdelt2_load  = (pd.concat([pd.read_csv(f) for f in gdelt2_files], ignore_index=True)
                if gdelt2_files else pd.DataFrame())

rss_files    = sorted(Path(RSS_DIR).glob('news_raw_*.csv'))
rss_load     = (pd.concat([pd.read_csv(f) for f in rss_files], ignore_index=True)
                if rss_files else pd.DataFrame())

yf_files     = sorted(Path(YF_DIR).glob('yf_news_*.csv'))
yf_load      = (pd.concat([pd.read_csv(f) for f in yf_files], ignore_index=True)
                if yf_files else (yf_all if 'yf_all' in globals() and not yf_all.empty
                                  else pd.DataFrame()))

av_load      = av_all if 'av_all' in globals() else pd.DataFrame()
wb_load      = wb_all if 'wb_all' in globals() else pd.DataFrame()

# Standardise all sources to the same schema
print('\n📋 Standardising all sources:')
parts = [
    standardize_df(gdelt2_load, 'GDELT2'),
    standardize_df(av_load,     'AlphaVantage'),
    standardize_df(wb_load,     'Wayback'),
    standardize_df(rss_load,    'RSS'),
    standardize_df(yf_load,     'YFinance'),
]

news_corpus = (
    pd.concat([p for p in parts if not p.empty], ignore_index=True)
    if any(not p.empty for p in parts) else pd.DataFrame()
)

# ── Two-pass deduplication ────────────────────────────────────────────────────
if not news_corpus.empty:
    # Drop very short titles (< 20 chars) — almost always navigation text or errors
    news_corpus = news_corpus[
        news_corpus['title'].notna() & (news_corpus['title'].str.len() > 20)
    ]

    # Pass 1: exact URL deduplication
    news_corpus = news_corpus.drop_duplicates(subset=['url'])

    # Pass 2: normalised title deduplication (catches same article from different domains)
    news_corpus['title_clean'] = news_corpus['title'].apply(normalize_title)
    news_corpus = news_corpus.drop_duplicates(subset=['title_clean']).drop(columns=['title_clean'])
    news_corpus = news_corpus.sort_values('published').reset_index(drop=True)

    # Assign source quality weights (used in downstream modelling)
    SOURCE_WEIGHT = {'RSS': 1.0, 'AlphaVantage': 1.2, 'GDELT2': 0.8, 'Wayback': 0.6, 'YFinance': 1.0}
    def map_weight(src):
        for k, v in SOURCE_WEIGHT.items():
            if k.lower() in str(src).lower():
                return v
        return 1.0

    news_corpus['source_weight'] = (
        news_corpus['source']
        .apply(map_weight)
    )

    print(f'\n✅ Unified corpus: {len(news_corpus):,} unique articles')

    if 'period' in news_corpus.columns:
        print(f'   Coverage: ', f'{news_corpus["period"].min()} → {news_corpus["period"].max()}')

else:
    print('⚠️ No data to build corpus — ', 'check that at least one source ran successfully.')


# Save unified corpus
CONSOLIDATED_RAW = os.path.join(CLEANED_DIR, 'news_corpus_all.csv')

news_corpus = news_corpus.reset_index()
news_corpus['index'] = news_corpus.apply(lambda row: str(row['index']) + "_" + row['source'], axis = 1)
news_corpus.rename(columns = {'index': 'idx'}, inplace = True)

display(news_corpus.head())
news_corpus.to_csv(CONSOLIDATED_RAW, index=False)
print(f'💾 Full corpus → {CONSOLIDATED_RAW}')


📋 Standardising all sources:
GDELT2      :  27,750 articles | 2017-01 → 2026-03
AlphaVantage:     305 articles | 2026-04 → 2026-05
Wayback     :   2,071 articles | 2007-01 → 2015-12
RSS         :     526 articles | 2026-04 → 2026-06

✅ Unified corpus: 12,940 unique articles
   Coverage:  2007-04 → 2026-06


,idx,source,published,period,title,summary,url,av_label,av_score,source_weight
0,0_Wayback:business-standard,Wayback:business-standard,2007-04-26 15:41:59,2007-04,ISRO successfully launches Italian satellite,NaN,https://web.archive.org/web/20070426154159/htt...,None,None,0.6
1,1_Wayback:business-standard,Wayback:business-standard,2007-05-09 04:48:21,2007-05,Growth rate likely to moderate this fiscal: Ec...,NaN,https://web.archive.org/web/20070509044821/htt...,None,None,0.6
2,2_Wayback:business-standard,Wayback:business-standard,2007-05-10 12:33:44,2007-05,"AAI to upgrade Kolkata airport, Chennai on hold",NaN,https://web.archive.org/web/20070510123344/htt...,None,None,0.6
3,3_Wayback:business-standard,Wayback:business-standard,2007-05-10 12:43:47,2007-05,DGCA norms to check overbooking by carriers,NaN,https://web.archive.org/web/20070510124347/htt...,None,None,0.6
4,4_Wayback:business-standard,Wayback:business-standard,2007-05-16 04:11:46,2007-05,Maran quits Cabinet under DMK pressure,NaN,https://web.archive.org/web/20070516041146/htt...,None,None,0.6


💾 Full corpus → /content/drive/MyDrive/Regime_Detection_Swagato_M25AI2107/Data/cleaned/news_corpus_all.csv


---
## Section 13 · [FINBERT] Sentiment Analysis on Full Corpus

**Why?**  
Raw article counts tell us *how much* news exists, but not *whether it is positive or negative*.  
Sentiment scores transform unstructured text into a quantitative signal that captures  
market anxiety, optimism, or uncertainty — which are theoretically linked to regime transitions.

**Model:** [ProsusAI/FinBERT](https://huggingface.co/ProsusAI/finbert) (Apache 2.0)  
Fine-tuned on 10,000 financial news sentences. Outputs three probabilities:  
`positive`, `negative`, `neutral`.

**Design decisions:**
- Articles with an `av_score` from Alpha Vantage skip FinBERT — the AV score is converted directly to the same triple format.  
- Results are checkpointed every 500 articles — if Colab crashes, the loop resumes from the last checkpoint.

**Saves:** `raw/sentiment/finbert_checkpoint.csv`


In [23]:
# ─────────────────────────────────────────────────────────────────────────────
# LOAD FINBERT MODEL
# ─────────────────────────────────────────────────────────────────────────────
# We use HuggingFace's pipeline API for clean batched inference.
# device=0 uses GPU if available; device=-1 falls back to CPU.
device = 0 if torch.cuda.is_available() else -1
print(f'⏳ Loading FinBERT on {"GPU" if device == 0 else "CPU"} ...')

finbert_pipe = pipeline(
    'text-classification',
    model=FINBERT_MODEL,
    device=device,
    top_k=None,           # Return all 3 labels, not just the top-1
    truncation=True,
    max_length=512,
)
print('✅ FinBERT model ready.')


⏳ Loading FinBERT on CPU ...


config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

✅ FinBERT model ready.


In [24]:
news_corpus = pd.read_csv(os.path.join(CLEANED_DIR, 'news_corpus_all.csv'))
news_corpus['published'] = pd.to_datetime(news_corpus['published'],errors='coerce')

SENT_CHECKPOINT = os.path.join(SENTIMENT_DIR,'finbert_checkpoint.csv')

# Helper: AlphaVantage → FinBERT-style probabilities
def av_label_to_scores(label, score):
    """
    Convert AlphaVantage sentiment into FinBERT-style (positive, negative, neutral) probabilities.
    """
    if pd.isna(label) or pd.isna(score):
        return np.nan, np.nan, np.nan
    label = str(label).lower().strip()
    score = float(score)
    if label in ['bullish', 'somewhat_bullish']:
        return score, 1.0 - score, 0.0
    elif label in ['bearish', 'somewhat_bearish']:
        return 0.0, score, 1.0 - score
    else:
        return 0.0, 0.0, 1.0

# Resume from checkpoint if available

if os.path.exists(SENT_CHECKPOINT):
    scored_df = pd.read_csv(SENT_CHECKPOINT)
    scored_idx = set(scored_df.loc[scored_df['positive'].notna(),'idx'].astype(str))
    print(f'✅ Checkpoint found: ', f'{len(scored_idx):,} articles already scored.')
else:
    scored_df = news_corpus[['idx', 'source', 'published', 'period', 'title', 'summary', 'av_label', 'av_score', 'source_weight']].copy()
    scored_df['positive'] = np.nan
    scored_df['negative'] = np.nan
    scored_df['neutral'] = np.nan
    scored_idx = set()
    print('🆕 Starting fresh sentiment run.')

# Recompute derived columns for already-scored rows
already = scored_df['positive'].notna()
if already.any():
    scored_df.loc[already, 'net_sentiment'] = (scored_df.loc[already, 'positive'].astype(float)-scored_df.loc[already, 'negative'].astype(float))
    scored_df.loc[already, 'sentiment_label'] = scored_df.loc[already,['positive', 'negative', 'neutral']].astype(float).idxmax(axis=1)

# Find remaining articles
pending = news_corpus[~news_corpus['idx'].astype(str).isin(scored_idx)].copy()
print(f'Articles remaining to score: {len(pending):,}')

# Parameters
CHECKPOINT_EVERY = 500
processed_since_save = 0

# Run scoring
for chunk_start in tqdm(range(0, len(pending), BATCH_SIZE),desc='FinBERT scoring'):
    chunk = pending.iloc[chunk_start:chunk_start + BATCH_SIZE]
    for _, row in chunk.iterrows():
        article_id = str(row['idx'])

        # Alpha Vantage already available
        if pd.notna(row.get('av_score')):
            pos, neg, neu = av_label_to_scores(row.get('av_label'),row.get('av_score'))

        # FinBERT scoring
        else:
            summary = row.get('summary', '')
            if pd.isna(summary):
                summary = ''
            text = f"{row['title']}. {summary}"
            try:
                preds = finbert_pipe([text[:512]])[0]
                scores = {p['label'].lower(): p['score'] for p in preds}

                pos = scores.get('positive', 0.0)
                neg = scores.get('negative', 0.0)
                neu = scores.get('neutral', 1.0)
            except Exception as e:
                print(f"⚠️ Error on article {article_id}: {e}")
                pos, neg, neu = 0.0, 0.0, 1.0

        # Update scored dataframe using article ID
        mask = scored_df['idx'].astype(str) == article_id

        scored_df.loc[mask, 'positive'] = pos
        scored_df.loc[mask, 'negative'] = neg
        scored_df.loc[mask, 'neutral'] = neu

        processed_since_save += 1

        # Periodic checkpoint
        if processed_since_save >= CHECKPOINT_EVERY:
            scored_df.to_csv(SENT_CHECKPOINT,index=False)
            print(f'💾 Checkpoint saved ', f'({processed_since_save:,} articles)')
            processed_since_save = 0

# Final derived metrics
scored_df['positive'] = pd.to_numeric(scored_df['positive'],errors='coerce')
scored_df['negative'] = pd.to_numeric(scored_df['negative'],errors='coerce')
scored_df['neutral'] = pd.to_numeric(scored_df['neutral'],errors='coerce')
scored_df['net_sentiment'] = (scored_df['positive'] - scored_df['negative'])
scored_df['sentiment_label'] = scored_df[['positive', 'negative', 'neutral']].idxmax(axis=1)

# Save outputs
scored_df.to_csv(SENT_CHECKPOINT,index=False)
FINAL_OUTPUT = os.path.join(CLEANED_DIR,'news_corpus_all_sentiment.csv')
scored_df.to_csv(FINAL_OUTPUT, index=False)

# Summary
display(scored_df[scored_df.columns.to_list()[scored_df.columns.to_list().index('idx'):]].head())

print(f'\n✅ Sentiment scoring complete: ', f'{len(scored_df):,} articles.')

print(f'💾 Checkpoint → {SENT_CHECKPOINT}')
print(f'💾 Final corpus → {FINAL_OUTPUT}')
print(f'💾 With-sentiment corpus → {os.path.join(CLEANED_DIR, "news_corpus_all_sentiment.csv")}')

✅ Checkpoint found:  12,946 articles already scored.
Articles remaining to score: 150


FinBERT scoring:   0%|          | 0/5 [00:00<?, ?it/s]

,idx,source,published,period,title,summary,av_label,av_score,positive,negative,neutral,net_sentiment,sentiment_label
0,0_Wayback:business-standard,Wayback:business-standard,2007-04-26 15:41:59,2007-04,ISRO successfully launches Italian satellite,NaN,NaN,NaN,0.629198,0.010511,0.360291,0.618687,positive
1,1_Wayback:business-standard,Wayback:business-standard,2007-05-09 04:48:21,2007-05,Growth rate likely to moderate this fiscal: Ec...,NaN,NaN,NaN,0.342548,0.066359,0.591093,0.276189,neutral
2,2_Wayback:business-standard,Wayback:business-standard,2007-05-10 12:33:44,2007-05,"AAI to upgrade Kolkata airport, Chennai on hold",NaN,NaN,NaN,0.026670,0.086725,0.886605,-0.060055,neutral
3,3_Wayback:business-standard,Wayback:business-standard,2007-05-10 12:43:47,2007-05,DGCA norms to check overbooking by carriers,NaN,NaN,NaN,0.066480,0.024968,0.908552,0.041512,neutral
4,4_Wayback:business-standard,Wayback:business-standard,2007-05-16 04:11:46,2007-05,Maran quits Cabinet under DMK pressure,NaN,NaN,NaN,0.013723,0.879480,0.106797,-0.865758,negative



✅ Sentiment scoring complete:  12,946 articles.
💾 Checkpoint → /content/drive/MyDrive/Regime_Detection_Swagato_M25AI2107/Data/raw/sentiment/finbert_checkpoint.csv
💾 Final corpus → /content/drive/MyDrive/Regime_Detection_Swagato_M25AI2107/Data/cleaned/news_corpus_all_sentiment.csv
💾 With-sentiment corpus → /content/drive/MyDrive/Regime_Detection_Swagato_M25AI2107/Data/cleaned/news_corpus_all_sentiment.csv


In [25]:
scored_df.groupby('source')['sentiment_label'].value_counts().unstack(fill_value=0).reset_index()

sentiment_label,source,negative,neutral,positive
0,AlJazeera,19,5,5
1,AlphaVantage,62,222,14
2,BBC_World,20,8,3
3,Bloomberg_Asia,20,14,22
4,Bloomberg_Economy,28,7,16
5,CNBC_World,25,9,10
6,ET_Economy,29,16,39
7,ET_Markets,50,14,50
8,EconomicTimes,1,1,6
9,FederalReserve,1,2,0


---
## Section 14 · [RISK-CATEGORY] Classify Articles & Build Monthly Sentiment Table

**Why?**  
Raw net sentiment conflates all news — a central bank rate hike (financial)  
and a flood (climate) would both suppress sentiment, but the *channel* matters  
for regime modelling. We decompose sentiment by category to separate these signals.

**Classification logic:**
- Each article is scored for keyword density across 5 categories (financial, political, climate, green_transition, esg).
- Primary category = highest-density non-ESG/non-green category above threshold.
- ESG and green_transition are always scored but kept as separate continuous features (§4.2.6).

**Saves:** `raw/sentiment/news_sentiment_monthly.csv`


In [26]:
# ─────────────────────────────────────────────────────────────────────────────
# RISK CLASSIFICATION FUNCTIONS
# ─────────────────────────────────────────────────────────────────────────────

def score_risk_category(text: str) -> dict:
    """Compute keyword density for each risk category.

    Density = (keyword_hits) / (word_count).
    Using density (not raw count) makes long and short articles comparable.
    """
    t     = text.lower()
    words = len(t.split()) or 1   # Avoid division by zero
    return {cat: sum(1 for kw in kws if kw in t) / words
            for cat, kws in RISK_KEYWORDS.items()}


def assign_category(scores: dict, threshold: float = 0.002) -> str:
    """Assign a primary risk category label from density scores.

    ESG and green_transition are excluded from primary labelling — they are
    always scored separately as sustainability features (paper §4.2.6).
    Multi-label articles (two categories within 10% of each other) get a
    pipe-delimited label (e.g. 'financial|political').
    """
    primary = {k: v for k, v in scores.items() if k in ('financial', 'political', 'climate')}
    mx_primary = max(primary.values()) if primary else 0
    if mx_primary < threshold:
        return 'other'
    winners = [c for c, sc in primary.items() if sc >= mx_primary * 0.9]
    return '|'.join(sorted(winners))


In [27]:
# ─────────────────────────────────────────────────────────────────────────────
# RUN: Score categories, assign labels, build monthly tables
# ─────────────────────────────────────────────────────────────────────────────

# Combine title + summary for scoring
text_col   = scored_df['title'].fillna('') + ' ' + scored_df['summary'].fillna('')
cat_scores = text_col.apply(score_risk_category)

# Attach all five category scores to the scored DataFrame
scored_df['score_financial']        = cat_scores.apply(lambda x: x['financial'])
scored_df['score_political']        = cat_scores.apply(lambda x: x['political'])
scored_df['score_climate']          = cat_scores.apply(lambda x: x['climate'])
scored_df['score_green_transition'] = cat_scores.apply(lambda x: x['green_transition'])
scored_df['score_esg']              = cat_scores.apply(lambda x: x['esg'])
scored_df['risk_category']          = cat_scores.apply(assign_category)

print("Post mapping all the categories:")
display(scored_df.head())

Post mapping all the categories:


,Unnamed: 0,idx,source,published,period,title,summary,av_label,av_score,positive,negative,neutral,net_sentiment,sentiment_label,score_financial,score_political,score_climate,score_green_transition,score_esg,risk_category
0,0,0_Wayback:business-standard,Wayback:business-standard,2007-04-26 15:41:59,2007-04,ISRO successfully launches Italian satellite,NaN,NaN,NaN,0.629198,0.010511,0.360291,0.618687,positive,0.0,0.200000,0.0,0.0,0.0,political
1,1,1_Wayback:business-standard,Wayback:business-standard,2007-05-09 04:48:21,2007-05,Growth rate likely to moderate this fiscal: Ec...,NaN,NaN,NaN,0.342548,0.066359,0.591093,0.276189,neutral,0.0,0.000000,0.0,0.0,0.0,other
2,2,2_Wayback:business-standard,Wayback:business-standard,2007-05-10 12:33:44,2007-05,"AAI to upgrade Kolkata airport, Chennai on hold",NaN,NaN,NaN,0.026670,0.086725,0.886605,-0.060055,neutral,0.0,0.000000,0.0,0.0,0.0,other
3,3,3_Wayback:business-standard,Wayback:business-standard,2007-05-10 12:43:47,2007-05,DGCA norms to check overbooking by carriers,NaN,NaN,NaN,0.066480,0.024968,0.908552,0.041512,neutral,0.0,0.000000,0.0,0.0,0.0,other
4,4,4_Wayback:business-standard,Wayback:business-standard,2007-05-16 04:11:46,2007-05,Maran quits Cabinet under DMK pressure,NaN,NaN,NaN,0.013723,0.879480,0.106797,-0.865758,negative,0.0,0.333333,0.0,0.0,0.0,political


In [28]:
# ─────────────────────────────────────────────────────────────────────────────
# BUILD MONTHLY SENTIMENT DATA TABLE
# ─────────────────────────────────────────────────────────────────────────────
# Phase 01B stores monthly aggregations from collected/scored articles only.
# Ratios, z-scores, rolling windows, imputation, and interaction features are
# intentionally left for Phase 02 / EDA.

scored_df = scored_df.copy()  # avoid SettingWithCopyWarning

# Ensure sentiment columns are numeric (may be object dtype after CSV round-trip)
for _col in ['positive', 'negative', 'neutral', 'net_sentiment']:
    if _col in scored_df.columns:
        scored_df[_col] = pd.to_numeric(scored_df[_col], errors='coerce')

scored_df['period'] = pd.to_datetime(scored_df['published'], errors='coerce').dt.strftime('%Y-%m')
scored_df = scored_df.dropna(subset=['period'])

# Core monthly aggregations
monthly = scored_df.groupby('period').agg(
        article_count      = ('title',          'count'),
        sources_count      = ('source',         'nunique'),
        positive_mean      = ('positive',        'mean'),
        negative_mean      = ('negative',        'mean'),
        neutral_mean       = ('neutral',         'mean'),
        net_sentiment_mean = ('net_sentiment',   'mean'),
        net_sentiment_std  = ('net_sentiment',   'std'),
        pct_positive       = ('sentiment_label', lambda x: (x == 'positive').mean()),
        pct_negative       = ('sentiment_label', lambda x: (x == 'negative').mean()),
        pct_neutral        = ('sentiment_label', lambda x: (x == 'neutral').mean()),
    ).reset_index()

print('\nAfter core monthly aggregation:')
display(monthly.head(3))

# Per-category monthly counts and mean sentiment. These are direct grouped summaries; derived ratios are computed later in EDA.
for cat in ['financial', 'political', 'climate']:
    mask    = scored_df['risk_category'].str.contains(cat, na=False)
    cat_grp = scored_df[mask].groupby('period').agg(**{f'count_{cat}': ('title', 'count'), f'sent_{cat}': ('net_sentiment', 'mean')}).reset_index()
    monthly = monthly.merge(cat_grp, on='period', how='left')

# Green-transition and ESG monthly grouped summaries.
gt_grp = scored_df[scored_df['score_green_transition'] > 0].groupby('period').agg(count_green_transition=('title', 'count'), sent_green_transition=('net_sentiment', 'mean')).reset_index()
monthly = monthly.merge(gt_grp, on='period', how='left')

# Coverage flags — which article sources contributed to each month.
for src_type, prefix in [('GDELT2','gdelt2'),('AlphaVantage','av'),('Wayback','wb'),('RSS','rss')]:
    src_periods = set(scored_df[scored_df['source'].str.startswith(src_type)]['period'].tolist())
    monthly[f'has_{prefix}'] = monthly['period'].isin(src_periods).astype(int)

# Save monthly sentiment data. GDELT1 tone is saved separately and merged as its
# own source column in [COMBINE]; no pre-2015 backfill is done in Phase 01B.
sent_path = os.path.join(SENTIMENT_DIR, 'news_sentiment_monthly.csv')
monthly.to_csv(sent_path, index=False)

print('\nAfter final monthly aggregation:')
display(monthly.head(3))

print(f'\n✅ Monthly sentiment data table: {monthly.shape}')
print(f'   Coverage: {monthly["period"].min()} → {monthly["period"].max()}')
print(f'💾 Saved: {sent_path}')



After core monthly aggregation:


,period,article_count,sources_count,positive_mean,negative_mean,neutral_mean,net_sentiment_mean,net_sentiment_std,pct_positive,pct_negative,pct_neutral
0,2007-04,1,1,0.629198,0.010511,0.360291,0.618687,NaN,1.0,0.0,0.0
1,2007-05,5,1,0.097149,0.255606,0.647245,-0.158457,0.429978,0.0,0.2,0.8
2,2007-07,1,1,0.392941,0.008767,0.598292,0.384175,NaN,0.0,0.0,1.0



After final monthly aggregation:


,period,article_count,sources_count,positive_mean,negative_mean,neutral_mean,net_sentiment_mean,net_sentiment_std,pct_positive,pct_negative,...,count_political,sent_political,count_climate,sent_climate,count_green_transition,sent_green_transition,has_gdelt2,has_av,has_wb,has_rss
0,2007-04,1,1,0.629198,0.010511,0.360291,0.618687,NaN,1.0,0.0,...,1.0,0.618687,NaN,NaN,NaN,NaN,0,0,1,0
1,2007-05,5,1,0.097149,0.255606,0.647245,-0.158457,0.429978,0.0,0.2,...,1.0,-0.865758,NaN,NaN,NaN,NaN,0,0,1,0
2,2007-07,1,1,0.392941,0.008767,0.598292,0.384175,NaN,0.0,0.0,...,NaN,NaN,NaN,NaN,1.0,0.384175,0,0,1,0



✅ Monthly sentiment data table: (215, 23)
   Coverage: 2007-04 → 2026-06
💾 Saved: /content/drive/MyDrive/Regime_Detection_Swagato_M25AI2107/Data/raw/sentiment/news_sentiment_monthly.csv


---
## Section 15 · [REGIME-EVENTS] Regime-Change Events Database

**Why?**  
Structural breaks in Indian equity markets frequently coincide with discrete events:  
elections, budget announcements, RBI policy decisions, and global macro shocks.  
Including binary event flags as features helps the regime classifier distinguish  
*event-driven* regime changes from *gradual drift*.

**Design principles:**
- All events are fetched **live** from citable sources (ECI, RBI, Wikipedia, etc.) — no hard-coded private lists.
- Records contain only event *types* — **no party names, leader names, or ideology** to avoid partisan bias.
- Each record includes a `source_url` for research reproducibility and citation.

| Event type | Source |
|------------|--------|
| Lok Sabha elections | ECI / Wikipedia |
| State elections (market-sensitive states) | ECI / Wikipedia |
| Union Budget dates | Ministry of Finance / Wikipedia |
| RBI MPC meetings | RBI press releases / Wikipedia |
| Global macro shocks | Wikipedia / IMF / NBER / WHO |
| Indian reforms & crises | RBI / SEBI / NSE / Wikipedia |

**Saves:** `raw/regime_events/regime_events.csv` · `raw/regime_events/regime_events_monthly.csv`


In [29]:
# ─────────────────────────────────────────────────────────────────────────────
# SHARED UTILITY — robust HTTP GET with retries
# ─────────────────────────────────────────────────────────────────────────────

HEADERS_WIKI = {
    'User-Agent'     : 'RegimeDetectionResearch/1.0 (academic)',
    'Accept-Language': 'en-US,en;q=0.9',
}

SESSION = requests.Session()
SESSION.headers.update(HEADERS_WIKI)

def safe_get(url: str, timeout: int = 20, retries: int = 3):
    """GET a URL with exponential back-off; return Response or None."""
    for attempt in range(retries):
        try:
            r = SESSION.get(url, timeout=timeout)
            r.raise_for_status()
            return r
        except requests.exceptions.RequestException as e:
            if attempt < retries - 1:
                print(f'   Retry {attempt+1}/{retries} failed. Retrying...')
                time.sleep(2 ** attempt)
            else:
                print(f'   ⚠️  GET failed ({url[:60]}...): {e}')
    return None

print('✅ Shared HTTP utility ready.')

✅ Shared HTTP utility ready.


In [30]:
# ─────────────────────────────────────────────────────────────────────────────
# EVENT FETCHER FUNCTIONS
# ─────────────────────────────────────────────────────────────────────────────
# Each function returns a list of 5-tuples: (period, event_type, description, source, source_url)

def fetch_loksabha_elections() -> list:
    """
    Fetch Lok Sabha election dates from Wikipedia.
    Falls back to a verified ECI list if parsing fails.
    """
    SOURCE_URL = 'https://en.wikipedia.org/wiki/List_of_Indian_general_elections'

    start_year = int(GPR_START[:4])
    end_year   = int(GPR_END[:4])
    known_months = {
        1999:'09',
        2004:'04',
        2009:'04',
        2014:'04',
        2019:'04',
        2024:'04'
    }

    events = []
    seen   = set()

    resp = safe_get(SOURCE_URL)
    if resp:
        soup = BeautifulSoup(resp.text, 'lxml')
        for table in soup.select('table.wikitable'):
            for row in table.find_all('tr')[1:]:
                cols = [td.get_text(strip=True) for td in row.find_all(['td', 'th'])]
                for cell in cols[:4]:
                    m = re.search(r'(19|20)\d{2}', cell)
                    if not m:
                        continue
                    yr = int(m.group())
                    if start_year <= yr <= end_year and yr not in seen:
                        seen.add(yr)
                        mo = known_months.get(yr, '04')
                        events.append((
                            f'{yr}-{mo}',
                            'ELECTION',
                            f'Lok Sabha General Election {yr}',
                            'ECI / Wikipedia',
                            SOURCE_URL
                        ))
                        break
        if events:
            events.sort(key=lambda x: x[0])
            print(f'   ✅ Wikipedia: {len(events)} Lok Sabha elections parsed.')
            return events


def fetch_state_elections() -> list:
    """
    Major market-sensitive state elections.

    These elections were selected because they are widely regarded as having
    significant national political and market relevance.

    States considered:
    - Uttar Pradesh
    - Maharashtra
    - West Bengal
    - Karnataka
    - Rajasthan
    - Madhya Pradesh
    - Gujarat
    - Telangana
    - Andhra Pradesh
    - Delhi

    Source:
    Election Commission of India (ECI)
    https://eci.gov.in/statistical-report/statistical-reports/
    """

    ECI_URL = "https://eci.gov.in/statistical-report/statistical-reports/"
    events = [
        # Uttar Pradesh
        ('2017-02', 'ELECTION', 'Uttar Pradesh State Election 2017', 'Election Commission of India', ECI_URL),
        ('2022-02', 'ELECTION', 'Uttar Pradesh State Election 2022', 'Election Commission of India', ECI_URL),
        # Gujarat
        ('2017-12', 'ELECTION', 'Gujarat State Election 2017', 'Election Commission of India', ECI_URL),
        ('2022-12', 'ELECTION', 'Gujarat State Election 2022', 'Election Commission of India', ECI_URL),
        # Karnataka
        ('2018-05', 'ELECTION', 'Karnataka State Election 2018', 'Election Commission of India', ECI_URL),
        ('2023-05', 'ELECTION', 'Karnataka State Election 2023', 'Election Commission of India', ECI_URL),
        # Madhya Pradesh + Rajasthan
        ('2018-12', 'ELECTION', 'MP and Rajasthan State Elections 2018', 'Election Commission of India', ECI_URL),
        ('2023-11', 'ELECTION', 'MP and Rajasthan State Elections 2023', 'Election Commission of India', ECI_URL),
        # Maharashtra
        ('2019-10', 'ELECTION', 'Maharashtra State Election 2019', 'Election Commission of India', ECI_URL),
        ('2024-11', 'ELECTION', 'Maharashtra State Election 2024', 'Election Commission of India', ECI_URL),
        # Delhi
        ('2020-02', 'ELECTION', 'Delhi State Election 2020', 'Election Commission of India', ECI_URL),
        # West Bengal
        ('2021-03', 'ELECTION', 'West Bengal State Election 2021', 'Election Commission of India', ECI_URL),
        # Telangana
        ('2023-11', 'ELECTION', 'Telangana State Election 2023', 'Election Commission of India', ECI_URL),
        # Andhra Pradesh
        ('2024-05', 'ELECTION', 'Andhra Pradesh State Election 2024', 'Election Commission of India', ECI_URL),
    ]

    start_year = int(GPR_START[:4])
    end_year   = int(GPR_END[:4])
    events = [e for e in events if start_year <= int(e[0][:4]) <= end_year]
    events.sort(key=lambda x: x[0])
    print(f"   ✅ State elections: {len(events)} events")
    return events

def fetch_union_budgets() -> list:
    """
    Union Budget presentation dates.

    Source:
    Ministry of Finance (India)
    https://dea.gov.in/union-budget

    Dates are curated and verified for reproducibility.
    """

    MOF_URL = "https://dea.gov.in/union-budget"
    budget_dates = {
        1999: "02",
        2000: "02",
        2001: "02",
        2002: "02",
        2003: "02",
        2004: "07",  # Interim Feb + Full Jul
        2005: "02",
        2006: "02",
        2007: "02",
        2008: "02",
        2009: "07",  # Interim Feb + Full Jul
        2010: "02",
        2011: "02",
        2012: "03",
        2013: "02",
        2014: "07",  # Interim Feb + Full Jul
        2015: "02",
        2016: "02",
        2017: "02",  # Budget moved from end-Feb to 1-Feb
        2018: "02",
        2019: "07",  # Interim Feb + Full Jul
        2020: "02",
        2021: "02",
        2022: "02",
        2023: "02",
        2024: "07",  # Interim Feb + Full Jul
        2025: "02",
        2026: "02",
    }

    interim_years = {2004, 2009, 2014, 2019, 2024}

    start_year = int(GPR_START[:4])
    end_year   = int(GPR_END[:4])

    events = []
    for yr, mo in sorted(budget_dates.items()):
        if not (start_year <= yr <= end_year):
            continue
        label = f"Union Budget FY{yr}-{str(yr + 1)[-2:]}"
        if yr in interim_years:
            label += " [Interim/General Election Year]"
        events.append(
            (
                f"{yr}-{mo}",
                "UNION_BUDGET",
                label,
                "Ministry of Finance India",
                MOF_URL,
            )
        )
    print(f"   ✅ Union Budgets: {len(events)} events")
    return events

def fetch_rbi_mpc_dates() -> list:
    """
    RBI monetary policy events.

    Monthly regime-detection version.

    Sources:
    - RBI Monetary Policy Department
    - RBI MPC schedules
    - RBI historical policy reviews

    Notes
    -----
    * Pre-Oct-2016: RBI policy reviews.
    * Oct-2016 onward: Monetary Policy Committee (MPC).
    * Monthly granularity is sufficient because the modelling
      framework operates at monthly frequency.
    """

    RBI_URL = "https://www.rbi.org.in"
    start_year = int(GPR_START[:4])
    end_year   = int(GPR_END[:4])

    events = []

    # PRE-MPC ERA
    pre_mpc_months = [1, 4, 7, 10]
    for yr in range(start_year, min(2016, end_year + 1)):
        for mo in pre_mpc_months:
            events.append(
                (
                    f"{yr}-{mo:02d}",
                    "RBI_POLICY",
                    f"RBI Policy Review {yr}",
                    "Reserve Bank of India",
                    RBI_URL,
                )
            )

    # MPC FORMATION
    if start_year <= 2016 <= end_year:
        events.append(
            (
                "2016-10",
                "RBI_POLICY",
                "MPC Formation and First Meeting",
                "Reserve Bank of India",
                RBI_URL,
            )
        )

    # MPC ERA
    mpc_months = [2, 4, 6, 8, 10, 12]
    for yr in range(max(2017, start_year), end_year + 1):
        for mo in mpc_months:
            events.append(
                (
                    f"{yr}-{mo:02d}",
                    "RBI_POLICY",
                    f"RBI MPC Meeting {yr}",
                    "Reserve Bank of India",
                    RBI_URL,
                )
            )

    # Deduplicate
    seen = set()
    unique = []
    for e in events:
        key = (e[0], e[1], e[2])
        if key not in seen:
            seen.add(key)
            unique.append(e)
    unique.sort(key=lambda x: x[0])
    print(f"   ✅ RBI policy events: {len(unique)}")
    return unique


def fetch_global_macro_shocks() -> list:
    """Return a curated list of global macro shocks with citable Wikipedia/IMF/NBER sources."""
    events = [
        ('2000-03','GLOBAL_SHOCK','Dot-com bubble peak and burst','NBER / Wikipedia',
         'https://en.wikipedia.org/wiki/Dot-com_bubble'),
        ('2001-09','GLOBAL_SHOCK','September 11 attacks — global financial shock','Wikipedia / NBER',
         'https://en.wikipedia.org/wiki/Economic_effects_of_the_September_11_attacks'),
        ('2003-03','GLOBAL_SHOCK','Iraq War commencement — geopolitical risk spike','Wikipedia / IMF',
         'https://en.wikipedia.org/wiki/Iraq_War'),
        ('2004-05','GLOBAL_SHOCK','US Fed rate hike cycle start (Greenspan era)','Federal Reserve FOMC',
         'https://www.federalreserve.gov/monetarypolicy/fomchistorical2004.htm'),
        ('2007-08','GLOBAL_SHOCK','US subprime mortgage crisis onset','NBER / Wikipedia',
         'https://en.wikipedia.org/wiki/Subprime_mortgage_crisis'),
        ('2008-09','GLOBAL_SHOCK','Lehman Brothers collapse — Global Financial Crisis','NBER / Wikipedia',
         'https://en.wikipedia.org/wiki/Bankruptcy_of_Lehman_Brothers'),
        ('2010-05','GLOBAL_SHOCK','European sovereign debt crisis peak (Greece bailout)','IMF / Wikipedia',
         'https://en.wikipedia.org/wiki/2010_European_sovereign_debt_crisis'),
        ('2013-05','GLOBAL_SHOCK','Fed taper tantrum — Bernanke taper announcement','Federal Reserve',
         'https://en.wikipedia.org/wiki/2013_taper_tantrum'),
        ('2015-08','GLOBAL_SHOCK','China yuan devaluation + Shanghai market crash','IMF / Wikipedia',
         'https://en.wikipedia.org/wiki/2015%E2%80%9316_Chinese_stock_market_turbulence'),
        ('2016-06','GLOBAL_SHOCK','Brexit referendum result','Wikipedia / IMF',
         'https://en.wikipedia.org/wiki/2016_United_Kingdom_European_Union_membership_referendum'),
        ('2018-10','GLOBAL_SHOCK','US-China trade war escalation (tariff round)','WTO / Wikipedia',
         'https://en.wikipedia.org/wiki/China%E2%80%93United_States_trade_war'),
        ('2020-03','GLOBAL_SHOCK','COVID-19 declared global pandemic by WHO','World Health Organization',
         'https://www.who.int/director-general/speeches/detail/who-director-general-s-opening-remarks-at-the-media-briefing-on-covid-19---11-march-2020'),
        ('2020-04','GLOBAL_SHOCK','WTI crude oil futures turn negative (demand collapse)','CME Group / Wikipedia',
         'https://en.wikipedia.org/wiki/2020_oil_price_war'),
        ('2021-11','GLOBAL_SHOCK','Omicron variant announced by WHO','World Health Organization',
         'https://www.who.int/news/item/26-11-2021-classification-of-omicron-(b.1.1.529)-sars-cov-2-variant-of-concern'),
        ('2022-02','GLOBAL_SHOCK','Russia invasion of Ukraine','United Nations / Wikipedia',
         'https://en.wikipedia.org/wiki/Russian_invasion_of_Ukraine'),
        ('2022-03','GLOBAL_SHOCK','US Fed aggressive rate hike cycle start','Federal Reserve FOMC',
         'https://www.federalreserve.gov/monetarypolicy/fomchistorical2022.htm'),
        ('2023-03','GLOBAL_SHOCK','Silicon Valley Bank collapse — US banking stress','FDIC / Wikipedia',
         'https://en.wikipedia.org/wiki/Collapse_of_Silicon_Valley_Bank'),
        ('2023-10','GLOBAL_SHOCK','Israel-Hamas conflict escalation','United Nations / Wikipedia',
         'https://en.wikipedia.org/wiki/2023_Hamas-led_attack_on_Israel'),
        ('2024-08','GLOBAL_SHOCK','Japan yen carry-trade unwind — global risk-off','Bank of Japan / Wikipedia',
         'https://en.wikipedia.org/wiki/2024_stock_market_crash'),
        ('2025-01','GLOBAL_SHOCK','US tariff escalation wave (Trump administration)','USTR / Wikipedia',
         'https://en.wikipedia.org/wiki/Trump_tariffs'),
    ]
    events = [e for e in events if int(GPR_START[:4]) <= int(e[0][:4]) <= int(GPR_END[:4])]
    print(f'   ✅ Global macro shocks: {len(events)} events')
    return events


def fetch_india_reforms() -> list:
    """Return key Indian economic reform events with Ministry / Wikipedia citations."""
    events = [
        ('2000-02','REFORM','Information Technology Act 2000 enacted',
         'Ministry of Electronics and IT, GoI','https://en.wikipedia.org/wiki/Information_Technology_Act,_2000'),
        ('2008-01','REFORM','SEBI tightens FII norms','SEBI India','https://www.sebi.gov.in'),
        ('2016-11','REFORM','Demonetisation of ₹500 and ₹1000 notes','Reserve Bank of India / Wikipedia',
         'https://en.wikipedia.org/wiki/2016_Indian_banknote_demonetisation'),
        ('2017-07','REFORM','GST implementation','Ministry of Finance India / Wikipedia',
         'https://en.wikipedia.org/wiki/Goods_and_Services_Tax_(India)'),
        ('2019-09','REFORM','Corporate income tax cut (22% from 30%)','Ministry of Finance India',
         'https://en.wikipedia.org/wiki/2019_Indian_corporate_tax_rate_cut'),
        ('2020-09','REFORM','Farm Laws enacted (later repealed)','Government of India / Wikipedia',
         'https://en.wikipedia.org/wiki/2020_Indian_agriculture_acts'),
        ('2021-11','REFORM','Farm Laws repealed following farmer protests','Government of India / Wikipedia',
         'https://en.wikipedia.org/wiki/2020_Indian_agriculture_acts'),
        ('2023-09','REFORM','India sovereign bonds included in JPMorgan EM Bond Index',
         'JPMorgan / Wikipedia','https://en.wikipedia.org/wiki/Government_securities_in_India'),
        ('2024-01','REFORM','India bonds included in Bloomberg EM Index','Bloomberg / Reuters',
         'https://www.reuters.com/markets/rates-bonds/india-bonds-set-join-bloomberg-em-index-january-2025-2024-03-08/'),
    ]
    events = [e for e in events if int(GPR_START[:4]) <= int(e[0][:4]) <= int(GPR_END[:4])]
    print(f'   ✅ India reforms: {len(events)} events')
    return events


def fetch_india_crises() -> list:
    """Return key Indian market crisis events with NSE / RBI / SEBI citations."""
    events = [
        ('2008-10','CRISIS','NSE/BSE circuit breakers triggered repeatedly (GFC impact)',
         'NSE India / Wikipedia','https://en.wikipedia.org/wiki/2008_financial_crisis#Impact_on_India'),
        ('2013-08','CRISIS','INR hits historic low vs USD during Fed taper tantrum',
         'Reserve Bank of India','https://en.wikipedia.org/wiki/2013_taper_tantrum'),
        ('2018-09','CRISIS','IL&FS default triggers NBFC liquidity crisis',
         'SEBI / Wikipedia','https://en.wikipedia.org/wiki/IL%26FS_financial_crisis'),
        ('2019-08','CRISIS','Yes Bank / PMC Bank stress — regulatory intervention',
         'Reserve Bank of India','https://en.wikipedia.org/wiki/Yes_Bank_crisis'),
        ('2020-03','CRISIS','Nifty 50 falls ~38% in COVID crash (Feb–Mar 2020)',
         'NSE India / Wikipedia','https://en.wikipedia.org/wiki/2020_stock_market_crash'),
        ('2022-06','CRISIS','Global crypto crash + risk-off sell-off in Indian equities',
         'BIS / Wikipedia','https://en.wikipedia.org/wiki/Cryptocurrency_crash_(2022)'),
        ('2024-06','CRISIS','Post-election volatility spike on surprise result',
         'NSE India','https://www.nseindia.com'),
    ]
    events = [e for e in events if int(GPR_START[:4]) <= int(e[0][:4]) <= int(GPR_END[:4])]
    print(f'   ✅ India crises: {len(events)} events')
    return events

print('✅ All regime-event fetcher functions defined.')


✅ All regime-event fetcher functions defined.


In [31]:
# ─────────────────────────────────────────────────────────────────────────────
# RUN: Fetch all event categories, assemble DataFrame, create binary feature table
# ─────────────────────────────────────────────────────────────────────────────
print('\n🔗 Fetching regime-change events from all sources...\n')

ls_events           = fetch_loksabha_elections()
print(ls_events[0])
state_events        = fetch_state_elections()
print(state_events[0])
budget_events       = fetch_union_budgets()
print(budget_events[0])
rbi_events          = fetch_rbi_mpc_dates()
print(rbi_events[0])
global_shock_events = fetch_global_macro_shocks()
print(global_shock_events[0])
reform_events       = fetch_india_reforms()
print(reform_events[0])
crisis_events       = fetch_india_crises()
print(crisis_events[0])

# Assemble into a flat DataFrame
all_raw = []
for evt_list in [ls_events, state_events, budget_events,
                 rbi_events, global_shock_events, reform_events, crisis_events]:
    for e in evt_list:
        all_raw.append({
            'period'     : e[0],
            'event_type' : e[1],
            'description': e[2],
            'source'     : e[3],
            'source_url' : e[4],
        })

events_df = pd.DataFrame(all_raw)
events_df['period'] = pd.PeriodIndex(events_df['period'], freq='M')
events_df = events_df.sort_values('period').reset_index(drop=True)

print("\n\nAfter merging all:")
display(events_df.head())

# Build monthly binary feature table (one column per event type)
all_periods    = pd.period_range(start=GPR_START, end=GPR_END, freq='M')
events_monthly = pd.DataFrame({'period': all_periods})

for etype in ['ELECTION', 'UNION_BUDGET', 'RBI_POLICY', 'REFORM', 'GLOBAL_SHOCK', 'CRISIS']:
    etype_periods = set(events_df[events_df['event_type'] == etype]['period'].tolist())
    events_monthly[f'event_{etype.lower()}'] = events_monthly['period'].isin(etype_periods).astype(int)

# Derived: total event count and high-event-month flag (≥2 event types in same month)
event_cols = [c for c in events_monthly.columns if c.startswith('event_')]
events_monthly['event_count']      = events_monthly[event_cols].sum(axis=1)
events_monthly['high_event_month'] = (events_monthly['event_count'] >= 2).astype(int)

# Save both the event catalogue and the monthly binary table
events_path         = os.path.join(EVENTS_DIR, 'regime_events.csv')
events_monthly_path = os.path.join(EVENTS_DIR, 'regime_events_monthly.csv')
events_df.to_csv(events_path, index=False)
events_monthly.to_csv(events_monthly_path, index=False)

print("\n\nAfter converting into monthly:")
display(events_monthly.head())

print(f'\n✅ Event catalogue: {events_df.shape}')
print(f'✅ Monthly binary features: {events_monthly.shape}')
print(f'💾 {events_path}')
print(f'💾 {events_monthly_path}')
print('\nEvent-type distribution:')
print(events_df['event_type'].value_counts().to_string())



🔗 Fetching regime-change events from all sources...

   ✅ Wikipedia: 5 Lok Sabha elections parsed.
('2004-04', 'ELECTION', 'Lok Sabha General Election 2004', 'ECI / Wikipedia', 'https://en.wikipedia.org/wiki/List_of_Indian_general_elections')
   ✅ State elections: 14 events
('2017-02', 'ELECTION', 'Uttar Pradesh State Election 2017', 'Election Commission of India', 'https://eci.gov.in/statistical-report/statistical-reports/')
   ✅ Union Budgets: 27 events
('2000-02', 'UNION_BUDGET', 'Union Budget FY2000-01', 'Ministry of Finance India', 'https://dea.gov.in/union-budget')
   ✅ RBI policy events: 125
('2000-01', 'RBI_POLICY', 'RBI Policy Review 2000', 'Reserve Bank of India', 'https://www.rbi.org.in')
   ✅ Global macro shocks: 20 events
('2000-03', 'GLOBAL_SHOCK', 'Dot-com bubble peak and burst', 'NBER / Wikipedia', 'https://en.wikipedia.org/wiki/Dot-com_bubble')
   ✅ India reforms: 9 events
('2000-02', 'REFORM', 'Information Technology Act 2000 enacted', 'Ministry of Electronics and IT

,period,event_type,description,source,source_url
0,2000-01,RBI_POLICY,RBI Policy Review 2000,Reserve Bank of India,https://www.rbi.org.in
1,2000-02,UNION_BUDGET,Union Budget FY2000-01,Ministry of Finance India,https://dea.gov.in/union-budget
2,2000-02,REFORM,Information Technology Act 2000 enacted,"Ministry of Electronics and IT, GoI",https://en.wikipedia.org/wiki/Information_Tech...
3,2000-03,GLOBAL_SHOCK,Dot-com bubble peak and burst,NBER / Wikipedia,https://en.wikipedia.org/wiki/Dot-com_bubble
4,2000-04,RBI_POLICY,RBI Policy Review 2000,Reserve Bank of India,https://www.rbi.org.in




After converting into monthly:


,period,event_election,event_union_budget,event_rbi_policy,event_reform,event_global_shock,event_crisis,event_count,high_event_month
0,2000-01,0,0,1,0,0,0,1,0
1,2000-02,0,1,0,1,0,0,2,1
2,2000-03,0,0,0,0,1,0,1,0
3,2000-04,0,0,1,0,0,0,1,0
4,2000-05,0,0,0,0,0,0,0,0



✅ Event catalogue: (207, 5)
✅ Monthly binary features: (315, 9)
💾 /content/drive/MyDrive/Regime_Detection_Swagato_M25AI2107/Data/raw/regime_events/regime_events.csv
💾 /content/drive/MyDrive/Regime_Detection_Swagato_M25AI2107/Data/raw/regime_events/regime_events_monthly.csv

Event-type distribution:
event_type
RBI_POLICY      125
UNION_BUDGET     27
GLOBAL_SHOCK     20
ELECTION         19
REFORM            9
CRISIS            7


---
## Section 16 · [FRED] FRED Macroeconomic Data — India (23 Series)

**Why?**  
News sentiment captures *market perception*, but regime transitions are also driven by  
*underlying macro conditions* — GDP growth, inflation, credit expansion, trade balance, etc.  
FRED provides a standardised, freely accessible API for these indicators.

**Monthly series (14):** bond yields, interbank rates, REER, USD/INR, CPI, WPI,  
M1, M3, IIP growth, manufacturing production, OECD CLI, exports, imports, share prices.

**Quarterly series (9, forward-filled to monthly):** real GDP growth, private credit,  
total credit/GDP, financial account balance, capacity utilisation, business confidence,  
private consumption, GFCF, residential property prices.

**Free API key:** https://fred.stlouisfed.org/docs/api/api_key.html

**Saves:** `raw/fred/fred_india_macro.csv` · `raw/fred/annual_slices/fred_india_YYYY.csv`


In [32]:
from fredapi import Fred

fred = Fred(api_key=FRED_API_KEY) if FRED_API_KEY else None

def fetch_fred_series(series_id: str, col_name: str, freq: str = 'M') -> pd.Series:
    """Fetch one FRED series and resample to month-end.

    Quarterly series are forward-filled up to 3 months so they align with
    the monthly news sentiment table. This is a standard macro-modelling
    convention for mixed-frequency data.

    Args:
        series_id: FRED series identifier (e.g. 'INDGDPRQPSMEI').
        col_name:  Descriptive snake_case column name for the output.
        freq:      'M' for monthly, 'Q' for quarterly (forward-filled).
    """
    if fred is None:
        print(f'   ⚠️  [{series_id}] skipped: FRED_API_KEY not set')
        return pd.Series(name=col_name, dtype=float)
    try:
        raw = fred.get_series(series_id)
        raw.index = pd.to_datetime(raw.index)
        raw = raw.dropna()
        if freq == 'Q':
            # Resample to month-end, then forward-fill up to 3 months
            raw = raw.resample('ME').last().ffill(limit=3)
        else:
            raw = raw.resample('ME').last()
        raw.name = col_name
        return raw
    except Exception as e:
        print(f'   ⚠️  [{series_id}] fetch error: {e}')
        return pd.Series(name=col_name, dtype=float)

print('✅ FRED client initialised.')
print(f'   API key status: {"OK" if FRED_API_KEY else "NOT SET ⚠️"}')


✅ FRED client initialised.
   API key status: OK


In [33]:
# ─────────────────────────────────────────────────────────────────────────────
# RUN: Fetch all 23 FRED series (with cache — only runs if file not on disk)
# ─────────────────────────────────────────────────────────────────────────────
FRED_CACHE = os.path.join(FRED_DIR, 'fred_india_macro.csv')

if os.path.exists(FRED_CACHE):
    fred_df = pd.read_csv(FRED_CACHE, index_col=0, parse_dates=True)
    display(fred_df.head())
    print(f'✅ Loaded FRED cache: {fred_df.shape} | '
          f'{fred_df.index.min().date()} → {fred_df.index.max().date()}')
else:
    if not FRED_API_KEY:
        print('⚠️  FRED_API_KEY not set. Add FRED_API_KEY to Colab Secrets and re-run this cell.')
        fred_df = pd.DataFrame()
    else:
        series_list = []

        print('Fetching 14 monthly series...')
        for sid, col in tqdm(FRED_SERIES_MONTHLY.items(), desc='FRED monthly'):
            s = fetch_fred_series(sid, col, freq='M')
            if not s.empty:
                series_list.append(s)
                print(f'   ✓ {sid:25s} → {col:35s} ({len(s):,} obs)')
            time.sleep(0.3)   # FRED free tier: ~120 requests/minute

        print('\nFetching 9 quarterly series (forward-filled to monthly)...')
        for sid, col in tqdm(FRED_SERIES_QUARTERLY.items(), desc='FRED quarterly'):
            s = fetch_fred_series(sid, col, freq='Q')
            if not s.empty:
                series_list.append(s)
                print(f'   ✓ {sid:25s} → {col:35s} ({len(s):,} obs after ffill)')
            time.sleep(0.3)

        fred_df = pd.concat(series_list, axis=1).sort_index()
        # Trim to project date range
        fred_df = fred_df.loc[
            pd.Timestamp(GPR_START + '-01') : pd.Timestamp(GPR_END + '-28')
        ]
        display(fred_df.head())
        fred_df.to_csv(FRED_CACHE)
        print(f'\n✅ FRED macro data collected: {fred_df.shape}')
        print(f'   Coverage: {fred_df.index.min().date()} → {fred_df.index.max().date()}')
        print(f'💾 Saved: {FRED_CACHE}')

# Coverage summary (bar chart via ASCII)
if not fred_df.empty:
    print('\n📊 FRED series data coverage (% non-null):')
    coverage = (fred_df.notna().sum() / len(fred_df) * 100).sort_values(ascending=False)
    for col, pct in coverage.items():
        bar = '█' * int(pct // 5)
        print(f'   {col:40s} {pct:5.1f}% {bar}')


,rate_10y_bond_yield,rate_3m_interbank,fx_reer,fx_usdinr_avg,inf_cpi_yoy,inf_wpi_index,mon_m3_supply,mon_m1_growth_mom,act_mfg_production_sa,act_iip_growth_yoy,...,mkt_share_price_growth_mom,gdp_real_growth_yoy,credit_private_nonfin_inr,credit_total_nonfin_pct_gdp,bop_financial_acct_net_usd,biz_capacity_util_mfg,biz_confidence_mfg,gdp_private_consumption_yoy,gdp_gfcf_yoy,asset_residential_property_price
2000-01-31,NaN,NaN,86.82,43.546175,2.619048,45.429953,1.094368e+13,1.553075,36.87540,4.885905,...,12.562294,NaN,10953.471,136.7,-8.495604e+08,NaN,NaN,NaN,NaN,NaN
2000-02-29,NaN,NaN,88.08,43.601858,3.614458,45.585642,1.111108e+13,0.903789,36.27760,4.428913,...,4.539829,NaN,10953.471,136.7,-8.495604e+08,NaN,NaN,NaN,NaN,NaN
2000-03-31,NaN,NaN,90.22,43.578392,4.830918,46.550911,1.124174e+13,-0.285839,37.81580,8.625035,...,-7.127486,NaN,10953.471,136.7,-8.495604e+08,NaN,NaN,NaN,NaN,NaN
2000-04-30,NaN,NaN,91.81,43.630951,5.542169,47.235941,1.154093e+13,-0.081845,37.90019,7.477540,...,-6.510592,NaN,11421.896,138.5,-2.632161e+09,0.9,24.2151,NaN,NaN,NaN
2000-05-31,NaN,NaN,93.09,43.950131,5.011933,47.267079,1.161394e+13,1.220139,38.21259,5.548784,...,-12.953912,NaN,11421.896,138.5,-2.632161e+09,0.9,24.2151,NaN,NaN,NaN


✅ Loaded FRED cache: (314, 23) | 2000-01-31 → 2026-02-28

📊 FRED series data coverage (% non-null):
   fx_reer                                  100.0% ████████████████████
   fx_usdinr_avg                            100.0% ████████████████████
   mon_m1_growth_mom                        100.0% ████████████████████
   mkt_share_price_growth_mom               100.0% ████████████████████
   lead_oecd_cli_amp_adj                    100.0% ████████████████████
   act_iip_growth_yoy                       100.0% ████████████████████
   trade_imports_usd_sa                      99.4% ███████████████████
   trade_exports_usd_sa                      99.4% ███████████████████
   credit_total_nonfin_pct_gdp               97.8% ███████████████████
   credit_private_nonfin_inr                 97.8% ███████████████████
   biz_capacity_util_mfg                     96.8% ███████████████████
   biz_confidence_mfg                        96.8% ███████████████████
   inf_cpi_yoy                            

In [34]:
# ─────────────────────────────────────────────────────────────────────────────
# Save annual FRED slices (one CSV per year, handy for quick EDA)
# ─────────────────────────────────────────────────────────────────────────────
if not fred_df.empty:
    annual_dir = os.path.join(FRED_DIR, 'annual_slices')
    os.makedirs(annual_dir, exist_ok=True)
    for year, grp in fred_df.groupby(fred_df.index.year):
        path = os.path.join(annual_dir, f'fred_india_{year}.csv')
        grp.to_csv(path)
    print(f'💾 Annual FRED slices → {annual_dir}/')
    print('✅ FRED collection complete.')


💾 Annual FRED slices → /content/drive/MyDrive/Regime_Detection_Swagato_M25AI2107/Data/raw/fred/annual_slices/
✅ FRED collection complete.


---
## Section 16B · [MARKET-DATA] India VIX · USD/INR · Gold — Historical Market Data

**Why?**  
Three additional market-state signals that meaningfully complement the macro/news features already collected:

| Series | Proxy for | Source |
|--------|-----------|--------|
| **India VIX** | Implied volatility of NIFTY 50 options → forward-looking fear/uncertainty | `yfinance` (`^INDIAVIX`) |
| **USD/INR** | Currency stress, capital-flow pressure, RBI intervention signal | `yfinance` (`USDINR=X`) |
| **Gold (USD/oz)** | Global safe-haven demand, inflation hedge, geopolitical risk proxy | `yfinance` (`GC=F` / `GLD`) |

All three series are resampled to **month-end** (last trading day) to match the project's
monthly cadence. Month-over-month returns and a 12-month rolling z-score are computed
for each so Phase 02 can use both levels and change signals without look-ahead bias.

**Saves:**
- `raw/other_market_data/india_vix_monthly.csv`
- `raw/other_market_data/usdinr_monthly.csv`
- `raw/other_market_data/gold_monthly.csv`
- `raw/other_market_data/other_market_data_combined.csv` ← all three merged


In [35]:
# ─────────────────────────────────────────────────────────────────────────────
# 16B-SETUP  fetch_yf_monthly() helper
# ─────────────────────────────────────────────────────────────────────────────
# MKT_START, MKT_END, and MARKET_DIR are defined in [CFG] (Section 3).
# yfinance is imported in [LIBS] (Section 2).

def fetch_yf_monthly(
    ticker: str,
    col_close: str,
    start: str = MKT_START,
    end: str   = MKT_END,
) -> pd.DataFrame:
    """
    Download daily OHLCV from yfinance, resample to month-end close,
    and compute MoM % return + 12-month rolling z-score.

    Args:
        ticker:    yfinance ticker symbol (e.g. '^INDIAVIX').
        col_close: Column name for the close price in the output.
        start:     Fetch start date string 'YYYY-MM-DD' (default: MKT_START from CFG).
        end:       Fetch end date string 'YYYY-MM-DD' (default: MKT_END from CFG).

    Returns:
        DataFrame indexed by monthly Period with columns:
          <col_close>              — month-end close price
          <col_close>_mom_pct      — month-over-month % change
          <col_close>_zscore12     — 12-month rolling z-score of the close
    """
    print(f'⬇️  Fetching {ticker} ({col_close}) …')
    try:
        raw = yf.download(
            ticker,
            start=start,
            end=end,
            interval='1d',
            auto_adjust=True,
            progress=False,
        )
        if raw.empty:
            print(f'   ⚠️  {ticker}: no data returned.')
            return pd.DataFrame()

        # Flatten MultiIndex columns produced by yfinance ≥0.2
        if isinstance(raw.columns, pd.MultiIndex):
            raw.columns = raw.columns.get_level_values(0)

        close = raw['Close'].dropna().rename(col_close)

        # Resample to month-end — takes the last trading day of each calendar month
        monthly = close.resample('ME').last().dropna()

        df = monthly.to_frame()
        df[f'{col_close}_mom_pct'] = df[col_close].pct_change() * 100       # MoM % change
        df[f'{col_close}_zscore12'] = (
            (df[col_close] - df[col_close].rolling(12).mean())
            / df[col_close].rolling(12).std()
        )                                                                     # 12-month z-score (NaN for first 11 months)

        df.index = df.index.to_period('M')
        df.index.name = 'period'
        print(f'   ✅ {ticker}: {len(df)} monthly obs | '
              f'{df.index.min()} → {df.index.max()}')
        return df

    except Exception as e:
        print(f'   ❌ {ticker}: fetch failed — {e}')
        return pd.DataFrame()


print('✅ [MARKET-DATA] fetch_yf_monthly() helper defined.')


✅ [MARKET-DATA] fetch_yf_monthly() helper defined.


In [36]:
# ─────────────────────────────────────────────────────────────────────────────
# 16B-RUN  Fetch India VIX · USD/INR · Gold (with per-series disk cache)
# ─────────────────────────────────────────────────────────────────────────────
# Each series is cached independently — a failure on one does not block others.
# Re-running this cell loads from cache; delete a cache file to force re-fetch.

# Initialise to empty so COMBINE step has a safe default even if fetches fail
vix_df      = pd.DataFrame()
usdinr_df   = pd.DataFrame()
gold_df     = pd.DataFrame()
market_combined = pd.DataFrame()

In [40]:
# ── 1. India VIX  ────────────────────────────────────────────────────────────
# ^INDIAVIX = NSE India Volatility Index (NIFTY 50 30-day implied vol).
# NSE launched India VIX in Mar 2009 — pre-2009 months will be NaN in the master table.
VIX_CACHE = os.path.join(MARKET_DIR, 'india_vix_monthly.csv')

if os.path.exists(VIX_CACHE):
    vix_df = pd.read_csv(VIX_CACHE, index_col=0)
    vix_df.index = pd.PeriodIndex(vix_df.index, freq='M')
    display(vix_df.head())
    print(f'\n✅ Loaded India VIX cache: {vix_df.shape} | '
          f'{vix_df.index.min()} → {vix_df.index.max()}')
else:
    vix_df = fetch_yf_monthly('^INDIAVIX', 'india_vix')
    if not vix_df.empty:
        vix_df.to_csv(VIX_CACHE)
        print(f'💾 Saved: {VIX_CACHE}')

,india_vix,india_vix_mom_pct,india_vix_zscore12
period,,,
2008-03,35.139999,NaN,NaN
2008-04,26.160000,-25.554922,NaN
2008-05,26.680000,1.987769,NaN
2008-06,30.959999,16.041974,NaN
2008-07,39.060001,26.162799,NaN



✅ Loaded India VIX cache: (217, 3) | 2008-03 → 2026-03


In [42]:
# ── 2. USD/INR  ──────────────────────────────────────────────────────────────
# USDINR=X = INR per 1 USD (higher value = weaker rupee = more currency stress).
USDINR_CACHE = os.path.join(MARKET_DIR, 'usdinr_monthly.csv')

if os.path.exists(USDINR_CACHE):
    usdinr_df = pd.read_csv(USDINR_CACHE, index_col=0)
    usdinr_df.index = pd.PeriodIndex(usdinr_df.index, freq='M')
    display(usdinr_df.head())
    print(f'\n✅ Loaded USD/INR cache: {usdinr_df.shape} | '
          f'{usdinr_df.index.min()} → {usdinr_df.index.max()}')
else:
    usdinr_df = fetch_yf_monthly('USDINR=X', 'usdinr')
    if not usdinr_df.empty:
        usdinr_df.to_csv(USDINR_CACHE)
        print(f'💾 Saved: {USDINR_CACHE}')

,usdinr,usdinr_mom_pct,usdinr_zscore12
period,,,
2003-12,45.529999,NaN,NaN
2004-01,45.004002,-1.155276,NaN
2004-02,45.195999,0.426623,NaN
2004-03,43.334000,-4.119833,NaN
2004-04,44.284000,2.192276,NaN



✅ Loaded USD/INR cache: (268, 3) | 2003-12 → 2026-03


In [44]:
# ── 3. Gold (USD per troy ounce)  ────────────────────────────────────────────
# GC=F = COMEX Gold Futures (front month) — best liquid proxy for spot gold.
# Falls back to GLD ETF (~1/10 oz per share, available Nov 2004+) if GC=F is sparse.
GOLD_CACHE = os.path.join(MARKET_DIR, 'gold_monthly.csv')

if os.path.exists(GOLD_CACHE):
    gold_df = pd.read_csv(GOLD_CACHE, index_col=0)
    gold_df.index = pd.PeriodIndex(gold_df.index, freq='M')
    display(usdinr_df.head())
    print(f'\n✅ Loaded Gold cache: {gold_df.shape} | '
          f'{gold_df.index.min()} → {gold_df.index.max()}')
else:
    gold_df = fetch_yf_monthly('GC=F', 'gold_usd')
    if gold_df.empty:
        print('   ↩️  GC=F sparse — falling back to GLD ETF (available Nov 2004+) …')
        gold_df = fetch_yf_monthly('GLD', 'gold_usd')
    if not gold_df.empty:
        gold_df.to_csv(GOLD_CACHE)
        print(f'💾 Saved: {GOLD_CACHE}')

,usdinr,usdinr_mom_pct,usdinr_zscore12
period,,,
2003-12,45.529999,NaN,NaN
2004-01,45.004002,-1.155276,NaN
2004-02,45.195999,0.426623,NaN
2004-03,43.334000,-4.119833,NaN
2004-04,44.284000,2.192276,NaN



✅ Loaded Gold cache: (308, 3) | 2000-08 → 2026-03


In [45]:
# ── 4. Merge all three into one combined file  ───────────────────────────────
market_frames = [df for df in [vix_df, usdinr_df, gold_df] if not df.empty]

if market_frames:
    market_combined = market_frames[0].copy()
    for other in market_frames[1:]:
        # outer join — keeps all months even where one series has no observation
        market_combined = market_combined.join(other, how='outer')

    MARKET_COMBINED_CACHE = os.path.join(MARKET_DIR, 'other_market_data_combined.csv')
    market_combined.to_csv(MARKET_COMBINED_CACHE)
    display(market_combined.head())
    print(f'\n✅ Market data combined: {market_combined.shape[0]} months × {market_combined.shape[1]} columns')
    print(f'   Columns   : {list(market_combined.columns)}')
    print(f'   Coverage  : {market_combined.index.min()} → {market_combined.index.max()}')
    print(f'💾 Saved     : {MARKET_COMBINED_CACHE}')

    # Data-quality summary: % non-null per column
    print('\n📊 Market data coverage (% non-null):')
    cov = (market_combined.notna().sum() / len(market_combined) * 100)
    for col, pct in cov.items():
        bar = '█' * int(pct // 5)
        print(f'   {col:30s} {pct:5.1f}% {bar}')

    display(market_combined.dropna(how='all').tail(6))
else:
    print('⚠️  No market data series fetched successfully — market_combined is empty.')
    print('   Check network access and try deleting any 0-byte cache files.')

,india_vix,india_vix_mom_pct,india_vix_zscore12,usdinr,usdinr_mom_pct,usdinr_zscore12,gold_usd,gold_usd_mom_pct,gold_usd_zscore12
period,,,,,,,,,
2000-08,NaN,NaN,NaN,NaN,NaN,NaN,278.299988,NaN,NaN
2000-09,NaN,NaN,NaN,NaN,NaN,NaN,273.600006,-1.688819,NaN
2000-10,NaN,NaN,NaN,NaN,NaN,NaN,264.899994,-3.179829,NaN
2000-11,NaN,NaN,NaN,NaN,NaN,NaN,270.100006,1.963010,NaN
2000-12,NaN,NaN,NaN,NaN,NaN,NaN,272.000000,0.703441,NaN



✅ Market data combined: 308 months × 9 columns
   Columns   : ['india_vix', 'india_vix_mom_pct', 'india_vix_zscore12', 'usdinr', 'usdinr_mom_pct', 'usdinr_zscore12', 'gold_usd', 'gold_usd_mom_pct', 'gold_usd_zscore12']
   Coverage  : 2000-08 → 2026-03
💾 Saved     : /content/drive/MyDrive/Regime_Detection_Swagato_M25AI2107/Data/raw/other_market_data/other_market_data_combined.csv

📊 Market data coverage (% non-null):
   india_vix                       70.5% ██████████████
   india_vix_mom_pct               70.1% ██████████████
   india_vix_zscore12              66.9% █████████████
   usdinr                          87.0% █████████████████
   usdinr_mom_pct                  86.7% █████████████████
   usdinr_zscore12                 83.4% ████████████████
   gold_usd                       100.0% ████████████████████
   gold_usd_mom_pct                99.7% ███████████████████
   gold_usd_zscore12               96.4% ███████████████████


,india_vix,india_vix_mom_pct,india_vix_zscore12,usdinr,usdinr_mom_pct,usdinr_zscore12,gold_usd,gold_usd_mom_pct,gold_usd_zscore12
period,,,,,,,,,
2025-10,12.150000,9.756097,-0.740937,88.634499,-0.078913,1.469684,3982.199951,3.681522,1.789479
2025-11,11.620000,-4.362138,-0.847345,89.364799,0.823947,1.632328,4218.299805,5.928880,1.823084
2025-12,9.480000,-18.416526,-1.442668,89.769402,0.452753,1.521967,4325.600098,2.543686,1.680682
2026-01,13.630000,43.776379,0.305769,91.781898,2.241852,1.967358,4713.899902,8.976785,1.893251
2026-02,13.700000,0.513571,0.344179,91.007401,-0.843846,1.317816,5230.500000,10.959081,2.075566
2026-03,26.799999,95.620435,2.743719,94.309502,3.628387,2.013608,4492.000000,-14.119109,0.823132


---
## Section 17 · [VERIFY] Sanity-Check Output Files on Drive

**Why?**  
Before running the expensive [COMBINE] step, we confirm that all expected output files  
exist and are non-empty. This catches silent failures (e.g. a section that ran but produced  
no data due to a rate limit or network error).


In [46]:
import glob

print('── Output file inventory on Google Drive ──\n')

# Check all CSVs under DATA_ROOT
all_csvs = sorted(glob.glob(os.path.join(DATA_ROOT, '**/*.csv'), recursive=True))

# Group by sub-folder for a clean summary
folder_counts = {}
for path in all_csvs:
    size_kb = os.path.getsize(path) / 1024
    rel     = os.path.relpath(path, DATA_ROOT)
    folder  = os.sep.join(rel.split(os.sep)[:2]) if os.sep in rel else rel
    folder_counts[folder] = folder_counts.get(folder, 0) + 1
    # Flag tiny files that may indicate empty outputs
    flag = ' ⚠️ EMPTY?' if size_kb < 1.0 else ''
    print(f'  {rel:60s}  {size_kb:8.1f} KB{flag}')

print(f'\nTotal CSV files : {len(all_csvs)}')
print('\nFiles per subfolder:')
for folder, count in sorted(folder_counts.items()):
    print(f'  {folder:50s} {count} file(s)')

# Key file existence checks
KEY_FILES = [
    os.path.join(GPR_DIR,       'gpr_monthly.csv'),
    os.path.join(CLEANED_DIR,   'news_corpus_all.csv'),
    os.path.join(SENTIMENT_DIR, 'finbert_checkpoint.csv'),
    os.path.join(SENTIMENT_DIR, 'news_sentiment_monthly.csv'),
    os.path.join(EVENTS_DIR,    'regime_events_monthly.csv'),
    os.path.join(FRED_DIR,      'fred_india_macro.csv'),
    os.path.join(CPU_DIR,       'cpu_monthly.csv'),
    os.path.join(MARKET_DIR,    'india_vix_monthly.csv'),
    os.path.join(MARKET_DIR,    'usdinr_monthly.csv'),
    os.path.join(MARKET_DIR,    'gold_monthly.csv'),
]
print('\n── Key file existence check ──')
for f in KEY_FILES:
    exists = os.path.exists(f)
    size   = os.path.getsize(f) / 1024 if exists else 0
    status = f'✅ EXISTS  ({size:.1f} KB)' if exists else '❌ MISSING'
    print(f'  {os.path.relpath(f, DATA_ROOT):55s} {status}')

print('\n✅ Section 17 complete — review any ❌ or ⚠️ before running [COMBINE].')


── Output file inventory on Google Drive ──

  cleaned/BANKNIFTY_15min_cleaned.csv                             1253.9 KB
  cleaned/BANKNIFTY_1min_cleaned.csv                             10565.5 KB
  cleaned/BANKNIFTY_30min_cleaned.csv                              640.7 KB
  cleaned/BANKNIFTY_5min_cleaned.csv                              3816.3 KB
  cleaned/BANKNIFTY_60min_cleaned.csv                              335.0 KB
  cleaned/BANKNIFTY_daily_cleaned.csv                              621.2 KB
  cleaned/NIFTY50_15min_cleaned.csv                               1255.0 KB
  cleaned/NIFTY50_1min_cleaned.csv                               10566.2 KB
  cleaned/NIFTY50_30min_cleaned.csv                                641.4 KB
  cleaned/NIFTY50_5min_cleaned.csv                                3817.9 KB
  cleaned/NIFTY50_60min_cleaned.csv                                335.2 KB
  cleaned/NIFTY50_daily_cleaned.csv                                601.5 KB
  cleaned/NIFTYNEXT50_15min_cleaned.csv    

---
## Section 18 · [COMBINE] Build Final GPR Feature Table

**Why?**  
All preceding sections produce separate CSV files. This final step merges them into  
a single **monthly data table** (`gpr_combined_monthly.csv`) — one row per calendar month,  
covering 2000-01 → 2026-03. Feature engineering is intentionally deferred to Phase 02.

**Merge order:**
1. GPR Index (global + India)
2. CPU Index
3. Monthly news sentiment aggregates
4. Risk-category distribution (financial / political / climate / green / ESG)
5. Regime-change event binary flags
6. Alpha Vantage monthly sentiment
7. GDELT 1.0 monthly tone (pre-2015 proxy)
8. Climate-Policy Uncertainty monthly source series
9. Market data — India VIX · USD/INR · Gold
10. FRED macroeconomic source series

**Saves:** `cleaned/gpr_combined_monthly.csv`


In [47]:
# ─────────────────────────────────────────────────────────────────────────────
# RELOAD ALL MONTHLY COMPONENTS FROM DISK
# ─────────────────────────────────────────────────────────────────────────────
# Phase 01B is a data collection notebook. This cell only aligns and saves
# collected monthly tables; derived features are created later in Phase 02/EDA.

def load_csv(path: str, name: str, **kwargs) -> pd.DataFrame:
    if os.path.exists(path):
        df = pd.read_csv(path, **kwargs)
        print(f'✅ Loaded {name}: {df.shape[0]:,} rows × {df.shape[1]} cols')
        return df
    print(f'⚠️  Missing {name}: {path}')
    return pd.DataFrame()

gpr_index      = load_csv(os.path.join(GPR_DIR, 'gpr_monthly.csv'), 'GPR monthly', index_col=0)
monthly        = load_csv(os.path.join(SENTIMENT_DIR, 'news_sentiment_monthly.csv'), 'monthly sentiment')
events_monthly = load_csv(os.path.join(EVENTS_DIR, 'regime_events_monthly.csv'), 'monthly events')
fred_df        = load_csv(os.path.join(FRED_DIR, 'fred_india_macro.csv'), 'FRED macro', index_col=0, parse_dates=True)
av_monthly     = load_csv(os.path.join(AV_DIR, 'av_monthly_sentiment.csv'), 'Alpha Vantage monthly')
gdelt1_tone    = load_csv(os.path.join(GDELT_DIR, 'gdelt1_monthly_tone.csv'), 'GDELT1 monthly tone')
cpu_monthly    = load_csv(os.path.join(CPU_DIR,       'cpu_monthly.csv'), 'CPU monthly')
market_data    = load_csv(os.path.join(MARKET_DIR,    'other_market_data_combined.csv'), 'Market data (VIX/USDINR/Gold)', index_col=0)

# ─────────────────────────────────────────────────────────────────────────────
# BUILD SKELETON: one row per month for the full project date range
# ─────────────────────────────────────────────────────────────────────────────
all_periods = pd.period_range(start=GPR_START, end=GPR_END, freq='M')
combined    = pd.DataFrame({'period': all_periods}).set_index('period')

def join_period_df(base: pd.DataFrame, df: pd.DataFrame, name: str, period_col: str = 'period') -> pd.DataFrame:
    if df.empty:
        print(f'⚠️  {name} unavailable; skipped.')
        return base
    work = df.copy()
    if period_col in work.columns:
        work[period_col] = pd.PeriodIndex(work[period_col].astype(str), freq='M')
        work = work.set_index(period_col)
    else:
        work.index = pd.PeriodIndex(work.index.astype(str), freq='M')
    work.index.name = 'period'
    overlap = [c for c in work.columns if c in base.columns]
    if overlap:
        work = work.drop(columns=overlap)
        print(f'   ℹ️  {name}: dropped {len(overlap)} duplicate columns.')
    if work.empty:
        print(f'   ℹ️  {name}: no new columns to merge.')
        return base
    print(f'✅ {name} merged: {work.shape[1]} columns')
    return base.join(work, how='left')

combined = join_period_df(combined, gpr_index, 'GPR monthly')
combined = join_period_df(combined, monthly, 'monthly sentiment')
combined = join_period_df(combined, events_monthly, 'monthly events')
combined = join_period_df(combined, av_monthly, 'Alpha Vantage monthly')
combined = join_period_df(combined, gdelt1_tone, 'GDELT1 monthly tone')
combined = join_period_df(combined, cpu_monthly, 'CPU monthly')

# Market data: India VIX · USD/INR · Gold (Section 16B)
if not market_data.empty:
    mkt = market_data.copy()
    mkt.index = pd.PeriodIndex(mkt.index.astype(str), freq='M')
    mkt.index.name = 'period'
    combined = join_period_df(combined, mkt, 'Market data (VIX/USDINR/Gold)')
else:
    print('⚠️  Market data unavailable; skipped.')

if not fred_df.empty:
    fred_p = fred_df.copy()
    fred_p.index = pd.to_datetime(fred_p.index, errors='coerce').to_period('M')
    fred_p.index.name = 'period'
    combined = join_period_df(combined, fred_p, 'FRED macro')
else:
    print('⚠️  FRED macro unavailable; skipped.')

# Save master monthly data table. No z-scores, rolling means, shock flags,
# composite scores, or interaction terms are created in Phase 01B.
combined = combined.reset_index()
combined['period'] = combined['period'].astype(str)
combined_path = os.path.join(CLEANED_DIR, 'gpr_combined_monthly.csv')
display(combined.head())
combined.to_csv(combined_path, index=False)

print(f'\n✅ Master monthly data table: {combined.shape}')
print(f'   Periods  : {combined["period"].min()} → {combined["period"].max()}')
print(f'   Columns  : {combined.shape[1]}')
print(f'💾 Saved   : {combined_path}')

# ── Column-group fill-rate report ─────────────────────────────────────────────
print('\n📊 Column fill-rate by feature group (% non-null across all months):')
groups = {
    'GPR'        : [c for c in combined.columns if c.startswith('gpr_')],
    'Sentiment'  : [c for c in combined.columns if any(c.startswith(p) for p in ['article_','positive','negative','neutral','net_sent','pct_','count_','sent_','sources_','has_'])],
    'Events'     : [c for c in combined.columns if c.startswith('event_') or c in ['event_count','high_event_month']],
    'AV'         : [c for c in combined.columns if c.startswith('av_')],
    'GDELT1'     : [c for c in combined.columns if c.startswith('gdelt1_')],
    'CPU'        : [c for c in combined.columns if c.startswith('cpu_')],
    'Market'     : [c for c in combined.columns if any(c.startswith(p) for p in ['india_vix','usdinr','gold_usd'])],
    'FRED'       : [c for c in combined.columns if any(c.startswith(p) for p in ['rate_','fx_','inf_','mon_','act_','lead_','trade_','mkt_','gdp_','credit_','bop_','biz_','asset_'])],
}
for grp, cols in groups.items():
    if not cols:
        continue
    pct = combined[cols].notna().mean().mean() * 100
    bar = '█' * int(pct // 5)
    print(f'   {grp:12s} ({len(cols):2d} cols)  {pct:5.1f}%  {bar}')

print('\nLast 6 months — non-null column count:')
print(combined[['period']].assign(non_null_cols=combined.notna().sum(axis=1)).tail(6).to_string(index=False))


✅ Loaded GPR monthly: 1,517 rows × 13 cols
✅ Loaded monthly sentiment: 215 rows × 23 cols
✅ Loaded monthly events: 315 rows × 9 cols
✅ Loaded FRED macro: 314 rows × 23 cols
✅ Loaded Alpha Vantage monthly: 2 rows × 4 cols
✅ Loaded GDELT1 monthly tone: 22 rows × 5 cols
✅ Loaded CPU monthly: 462 rows × 4 cols
✅ Loaded Market data (VIX/USDINR/Gold): 308 rows × 9 cols
✅ GPR monthly merged: 13 columns
✅ monthly sentiment merged: 22 columns
✅ monthly events merged: 8 columns
✅ Alpha Vantage monthly merged: 3 columns
✅ GDELT1 monthly tone merged: 4 columns
✅ CPU monthly merged: 3 columns
✅ Market data (VIX/USDINR/Gold) merged: 9 columns
✅ FRED macro merged: 23 columns


,period,gpr_global,gpr_global_threats,gpr_global_acts,gpr_global_hist,gpr_global_hist_threats,gpr_global_hist_acts,gpr_share,gpr_news_count,gpr_global_noew,...,mkt_share_price_growth_mom,gdp_real_growth_yoy,credit_private_nonfin_inr,credit_total_nonfin_pct_gdp,bop_financial_acct_net_usd,biz_capacity_util_mfg,biz_confidence_mfg,gdp_private_consumption_yoy,gdp_gfcf_yoy,asset_residential_property_price
0,2000-01,64.457809,65.573601,64.250412,52.966461,60.844524,49.488827,1.933337,33362.0,76.979279,...,12.562294,NaN,10953.471,136.7,-8.495604e+08,NaN,NaN,NaN,NaN,NaN
1,2000-02,63.541721,57.651360,68.082169,54.460442,58.056553,52.426670,1.905860,32951.0,72.581642,...,4.539829,NaN,10953.471,136.7,-8.495604e+08,NaN,NaN,NaN,NaN,NaN
2,2000-03,50.101986,55.030735,40.816406,39.381897,52.421207,27.266571,1.502751,36533.0,67.415253,...,-7.127486,NaN,10953.471,136.7,-8.495604e+08,NaN,NaN,NaN,NaN,NaN
3,2000-04,48.682739,53.304905,40.917027,39.660149,53.651951,27.073410,1.460182,33352.0,70.115562,...,-6.510592,NaN,11421.896,138.5,-2.632161e+09,0.9,24.2151,NaN,NaN,NaN
4,2000-05,79.482880,77.269699,83.227814,60.203789,68.131752,56.935181,2.383997,35193.0,89.125244,...,-12.953912,NaN,11421.896,138.5,-2.632161e+09,0.9,24.2151,NaN,NaN,NaN



✅ Master monthly data table: (315, 86)
   Periods  : 2000-01 → 2026-03
   Columns  : 86
💾 Saved   : /content/drive/MyDrive/Regime_Detection_Swagato_M25AI2107/Data/cleaned/gpr_combined_monthly.csv

📊 Column fill-rate by feature group (% non-null across all months):
   GPR          (13 cols)  100.0%  ████████████████████
   Sentiment    (22 cols)   59.3%  ███████████
   Events       ( 8 cols)  100.0%  ████████████████████
   AV           ( 3 cols)    0.0%  
   GDELT1       ( 4 cols)    7.0%  █
   CPU          ( 3 cols)   98.1%  ███████████████████
   Market       ( 9 cols)   82.6%  ████████████████
   FRED         (23 cols)   85.7%  █████████████████

Last 6 months — non-null column count:
 period  non_null_cols
2025-10             64
2025-11             63
2025-12             63
2026-01             59
2026-02             59
2026-03             51


---
## ✅ Phase 01B Complete — Output File Summary

| File | Description | Source | Coverage |
|------|-------------|--------|----------|
| `raw/gpr/gpr_global_raw.xls` | Raw GPR download | Caldara & Iacoviello (2022) | 1985–2026 |
| `raw/gpr/gpr_monthly.csv` | GPR index (global + India) | Caldara & Iacoviello (2022) | 1985–2026 |
| `raw/rss/news_raw_YYYYMM.csv` | Monthly RSS articles | RSS feeds (27 sources) | ~2018–2026 |
| `raw/yfinance/yfinance_news.csv` | yfinance ticker news | Yahoo Finance | ~2020–2026 |
| `raw/gdelt/gdelt2_YYYY-MM.csv` | GDELT 2.0 monthly windows | GDELT Project | Feb 2015–2026 |
| `raw/gdelt/gdelt1_monthly_tone.csv` | GDELT 1.0 tone proxy | GDELT 1.0 | 2000–2015 |
| `raw/alphavantage/av_monthly_sentiment.csv` | Monthly AV sentiment | Alpha Vantage | ~2 years |
| `raw/wayback/wayback_titles.csv` | Wayback archived titles | Wayback CDX | 2006–2015 |
| `raw/fred/fred_india_macro.csv` | FRED macro (23 series) | FRED API | 2000–2026 |
| `raw/regime_events/regime_events.csv` | Event catalogue | ECI/RBI/MoF/Wikipedia | 2000–2026 |
| `raw/regime_events/regime_events_monthly.csv` | Monthly binary flags | Derived | 2000–2026 |
| `raw/sentiment/finbert_checkpoint.csv` | FinBERT article scores | ProsusAI/finbert | All articles |
| `raw/sentiment/news_sentiment_monthly.csv` | Monthly sentiment table | All sources | 2000–2026 |
| `raw/climate_policy/cpu_monthly.csv` | Raw monthly CPU index | Baker-Bloom-Davis | ~1990–2026 |
| **`cleaned/news_corpus_all.csv`** | **Unified news corpus** | **All sources** | **2000–2026** |
| `raw/other_market_data/india_vix_monthly.csv` | India VIX monthly close + MoM + z-score | yfinance (^INDIAVIX) | 2009–2026 |
| `raw/other_market_data/usdinr_monthly.csv` | USD/INR monthly close + MoM + z-score | yfinance (USDINR=X) | 2000–2026 |
| `raw/other_market_data/gold_monthly.csv` | Gold USD/oz monthly close + MoM + z-score | yfinance (GC=F / GLD) | 2000–2026 |
| `raw/other_market_data/other_market_data_combined.csv` | VIX + USD/INR + Gold merged | yfinance | 2000–2026 |
| **`cleaned/gpr_combined_monthly.csv`** | **MASTER monthly data table — GPR + sentiment + events + other market data + FRED macro (no derived features)** | **All above** | **2000–2026** |

---
